# Ensemble films: Film VI and E1–E7

The films of [`05_modeling/ensemble-claude.ipynb`](../05_modeling/ensemble-claude.ipynb), in the GRIDSTRESS style of films
I–V (`regression-models-hari.ipynb` §11 and `Hari_Grid-Stress_Film-V_Summary.ipynb`).

**Input: `data/films/film_inputs.joblib`**, written by §8 of the ensemble notebook. If the file is
missing, §1 runs the ensemble notebook first (`RUN_ENSEMBLE_IF_MISSING`, a few minutes); after that
this notebook reads only that file, fits nothing and loads no model save, so it starts in seconds.
After a change in the ensemble's §1.3, re-run its §8 (or delete the file) to refresh the inputs.

| film | shows |
|---|---|
| Film VI · the second glass | the summary in five acts: the field, the regime weights, the adaptive weights, the test year month by month, the verdict |
| E1 · the duel | as film I: every test day, a sliding week, the hourly gain, the race of the expanding MAE and the calendar of verdicts |
| E2 · hard-case chase | as film II: the six clearest wins and the six clearest losses, hour by hour |
| E3 · the mechanism | as film III: the opponent's day error against what the ensemble changes |
| E4 · one national day | as film IV: the day with the largest edge, hour by hour |
| E5 · the claim | §8 of hari: every ensemble's daily edge with its 95 % bootstrap interval, and when it is certain |
| E6 · the clock | §6, §9 of hari: by local hour, the opponent's bias against what the ensemble changes |
| E7 · the risk days | slide 13 of the deck: every high-risk and below-zero test day, caught, missed or false alarm, ensemble against SMARD |

E1–E7 are a **duel** between the ensemble and one opponent (`DUEL_AGAINST`): SMARD's official
day-ahead by default, as in films I–IV, or the best single member. Film VI is always set against
the best member, the reference spec 08 asks the ensemble to beat.

## How to use this notebook

1. Run it top to bottom. It draws contact sheets of key frames, then renders the mp4s to
   `data/films/` (`RENDER_FILMS = True`, the default). The first run takes several minutes and needs
   ffmpeg (`imageio-ffmpeg` or a system ffmpeg); later runs render only films that are out of date.
2. `FILMS_TO_RENDER` picks which films; `RENDER_FILMS = False` keeps to the contact sheets.
3. §5 plays the films here in the notebook (`SHOW_FILMS`), including films rendered in an earlier
   session.

## <a id="sec-1"></a>1 Setup and the inputs

In [ ]:
import io
import json
import os
import subprocess
import sys
import time
from pathlib import Path

import joblib
import matplotlib.animation as animation
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, Video, display

# Find the repository's data/ folder: the first folder, from here upwards, whose data/ holds smard.csv.
# Set the environment variable GRIDSTRESS_DATA to point at a data/ folder elsewhere.
candidates = [Path(os.environ["GRIDSTRESS_DATA"])] if os.environ.get("GRIDSTRESS_DATA") else []
candidates += [folder / "data" for folder in [Path.cwd(), *Path.cwd().parents]]
DATA_DIR = next((path for path in candidates if (path / "smard.csv").is_file()), None)
if DATA_DIR is None:
    raise RuntimeError(
        f"No data/smard.csv found in {Path.cwd()} or any parent folder. Run the notebook from inside the "
        "repository, or set GRIDSTRESS_DATA to the data/ folder that holds smard.csv."
    )

FILM_INPUTS = DATA_DIR / "films" / "film_inputs.joblib"
# Without film inputs, run the team's ensemble notebook first (about 4 minutes); its §8 writes them. It runs
# in its own kernel; the executed copy goes to data/films/, so the committed notebook stays unchanged.
RUN_ENSEMBLE_IF_MISSING = True
ENSEMBLE_NOTEBOOK = DATA_DIR.parent / "notebooks" / "05_modeling" / "ensemble-claude.ipynb"

if not FILM_INPUTS.exists() and RUN_ENSEMBLE_IF_MISSING:
    if not ENSEMBLE_NOTEBOOK.exists():
        raise FileNotFoundError(f"{ENSEMBLE_NOTEBOOK} not found: set ENSEMBLE_NOTEBOOK to the ensemble notebook")
    print(f"{FILM_INPUTS.name} not found: running {ENSEMBLE_NOTEBOOK.name} first (a few minutes) ...", flush=True)
    started = time.perf_counter()
    command = [sys.executable, "-m", "jupyter", "nbconvert", "--to", "notebook", "--execute", "--output-dir", str(FILM_INPUTS.parent),
               "--output", "ensemble-claude.executed.ipynb", "--ExecutePreprocessor.timeout=3600", str(ENSEMBLE_NOTEBOOK)]
    # the ensemble reads the same data/ folder as this notebook
    FILM_INPUTS.parent.mkdir(parents=True, exist_ok=True)
    done = subprocess.run(command, cwd=ENSEMBLE_NOTEBOOK.parent, capture_output=True, text=True,
                          env={**os.environ, "GRIDSTRESS_DATA": str(DATA_DIR)})
    if done.returncode != 0:
        # nbconvert ends its log with the cell that failed and its error: show that part
        raise RuntimeError(f"{ENSEMBLE_NOTEBOOK.name} stopped, so there are no film inputs. Its error:\n\n"
                           + "\n".join(done.stderr.strip().splitlines()[-25:]))
    print(f"  done in {time.perf_counter() - started:.0f} s; its outputs are in data/films/ensemble-claude.executed.ipynb")

if not FILM_INPUTS.exists():
    raise FileNotFoundError(
        f"{FILM_INPUTS.relative_to(DATA_DIR.parent).as_posix()} not found. Run notebooks/05_modeling/{ENSEMBLE_NOTEBOOK.name} to the end "
        "once: its §8 writes the film inputs."
    )
INPUTS = joblib.load(FILM_INPUTS)
if INPUTS.get("format") != 1:
    raise ValueError(f"{FILM_INPUTS.name} has format {INPUTS.get('format')!r}, this notebook reads format 1: "
                     "re-run §8 of ensemble-claude.ipynb")
FILM = INPUTS["film"]

# The team palette of streamlit/viz_helpers.py, applied here so inputs written by any version agree
PALETTE_MODELS = {
    "random_forest_hybrid": "#12989A", "lgbm_direct": "#2E64B0", "linear_direct": "#C27C00",
    "xgb_hybrid": "#9A5FA8", "lgbm_hybrid": "#C8553D", "xgb_direct": "#5566C2", "sarimax_fourier": "#A1C8CA",
    "smard": "#6B6E80", "seasonal_naive": "#A3A7B6", "actual": "#1B3C65",
}
PALETTE_ENSEMBLE = "#7A2A06"
FILM["member_color"] = {key: PALETTE_MODELS.get(key, color) for key, color in FILM["member_color"].items()}
FILM["member_color"]["actual"] = PALETTE_MODELS["actual"]  # residual load: the palette's ink
FILM["hero_color"] = PALETTE_ENSEMBLE
# The other ensembles in shades of the ensemble colour (the one shown keeps the full colour)
SHADES = {"equal_mean": "#C9937A", "weighted": "#A65A3A", "regime_weighted": "#7A2A06", "adaptive": "#4E1B04", "stacking": "#8C6D62"}
ensemble_colour = lambda method: PALETTE_ENSEMBLE if method == FILM.get("hero_method") else SHADES.get(method, "#8C6D62")
for method, meta in FILM.get("ensemble_meta", {}).items():
    meta["color"] = ensemble_colour(method)
if "verdict" in FILM:
    FILM["verdict"]["color"] = [
        ensemble_colour(model.removeprefix("ensemble_")) if model.startswith("ensemble_") else PALETTE_MODELS.get(model, colour)
        for (model, _), colour in zip(FILM["verdict"].index, FILM["verdict"]["color"])
    ]
DAY_NAMES = FILM["day_names"]
SEED = FILM["seed"]

print(f"film inputs from {INPUTS['source']}, saved {pd.Timestamp(INPUTS['saved_at']):%d %b %Y %H:%M}")
for name, span in INPUTS["data"].items():
    print(f"  run on {name}: {span['rows']:,} rows, {span['start']} .. {span['end']}")
print(f"  ensemble shown: {FILM['hero_label']} ({FILM['split']}, {FILM['hero_pick']}) · best member: {FILM['best_label']}")
print(f"  test year {FILM['test'][0]:%d %b %Y} .. {FILM['test'][1]:%d %b %Y} · {len(FILM['hourly']):,} common test hours")

### 1.1 Settings and the film frame

**Every number on screen is read from the film inputs**; nothing is typed in, nothing is
illustrative, so the film stays correct after a change in §1.3. The ensemble the film follows is
`PLOT_ROW` of §5.6 (with `PLOT_METHOD = None`: the enabled ensemble with the lowest **test** MAE, a
presentation pick like spec 09's, not a choice that feeds anything).

**Rendering.** §1.1 sets which films are rendered (`FILMS_TO_RENDER`) and how they are played
here (`SHOW_FILMS`); §4 renders, §5 plays..

In [ ]:
RENDER_FILMS = True    # True: write the mp4s that are missing or out of date (minutes); False: contact sheets only
FORCE_RENDER = False   # True: render again even when an mp4 is up to date (after a change in the drawing code)
FILMS_TO_RENDER = ["film_VI_the_second_glass", "film_E1_the_duel", "film_E2_hard_case_chase",
                   "film_E3_the_mechanism", "film_E4_one_national_day", "film_E5_the_claim", "film_E6_the_clock",
                   "film_E7_the_risk_days"]
SHOW_FILMS = "auto"    # play the films inside the notebook (§5): "link" plays data/films/*.mp4 from disk,
                       # "embed" stores them in the notebook (adds megabytes), "auto": embed in VS Code,
                       # which cannot play linked files, link elsewhere; None: off
FILM_FPS = 12          # frames per second; the film's length in seconds does not change
FILM_DIR = DATA_DIR / "films"  # data/ is gitignored
TIE_MWH = 50.0         # a day's MAE difference within ±TIE_MWH counts as a tie (as in film I)
E1_WINDOW = None       # E1 only: (first day, last day), e.g. ("2025-12-01", "2026-02-28"); None: the whole test year
DUEL_AGAINST = "smard" # E1–E7: "smard", the official day-ahead (as films I–IV), or "best", the best member

FILM_SIZE, FILM_DPI = (12.8, 7.2), 100  # 1280 × 720, like films I–IV

# The house palette of EDA-hari.ipynb; member and ensemble colours come from STYLE and METHODS
# The team palette (streamlit/viz_helpers.py): ink, muted, surfaces, grid, accent; teal = ensemble ahead,
# vermillion = opponent ahead, lavender = a tie
FILM_STYLE = {
    "ink": "#1B3C65", "muted": "#5C6275", "faint": "#A3A7B6", "grid": "#E4E6EE",
    "paper": "#FFFFFF", "card": "#F0F1F5", "border": "#E4E6EE", "accent": "#E7A118",
    "kicker": "#12989A", "kicker2": "#2E64B0",
    "win": "#12989A", "loss": "#C8553D", "tie": "#CCD3EB",
}
STAGE = [0.045, 0.10, 0.68, 0.70]  # left, bottom, width, height of the film's main area


def ease(t):
    """Cubic ease-out on [0, 1]."""
    t = min(max(t, 0.0), 1.0)
    return 1 - (1 - t) ** 3


def seg(t, start, end):
    """The local progress of an element that enters between `start` and `end` of an act."""
    return min(max((t - start) / (end - start), 0.0), 1.0)


def signed(value, digits=0, unit=""):
    """+1,234 or −1,234 with a true minus sign."""
    if pd.isna(value):
        return "–"
    text = f"{abs(value):,.{digits}f}"
    sign = "" if float(text.replace(",", "")) == 0 else "+" if value > 0 else "\u2212"
    return sign + text + (f" {unit}" if unit else "")


def fitted(text, size, width_px, bold=False):
    """A font size at most `size` that keeps `text` within `width_px` (DejaVu Sans, rough widths)."""
    per_char = (0.88 if bold else 0.80) * size * FILM_DPI / 100
    return min(size, width_px / max(per_char * len(text), 1e-9) * size)


def chrome(fig, act, headline, status, cards, caption, film="FILM VI · THE SECOND GLASS"):
    """Kicker, act label, headline, status line, four stat cards and the caption."""
    fig.patch.set_facecolor(FILM_STYLE["paper"])
    fig.text(0.035, 0.955, "GRIDSTRESS", color=FILM_STYLE["kicker"], fontsize=9.5, fontweight="bold")
    fig.text(0.112, 0.955, f"/  {film}", color=FILM_STYLE["kicker2"], fontsize=9.5, fontweight="bold")
    if act:
        fig.text(0.965, 0.955, act, color=FILM_STYLE["muted"], fontsize=8.5, ha="right")
    fig.text(0.035, 0.893, headline, color=FILM_STYLE["ink"], fontsize=19, fontweight="bold")
    if status:
        fig.text(0.035, 0.852, status, color=FILM_STYLE["muted"], fontsize=10)
    x0, width, height = 0.755, 0.21, 0.155
    for i, (title, value, sub, color) in enumerate(cards):
        top = 0.80 - i * 0.175
        fig.patches.append(mpatches.Rectangle(
            (x0, top - height), width, height, transform=fig.transFigure,
            facecolor=FILM_STYLE["card"], edgecolor=FILM_STYLE["border"], linewidth=1.0))
        fig.text(x0 + 0.012, top - 0.032, title, color=FILM_STYLE["muted"], fontsize=7.4, fontweight="bold")
        room = (width - 0.024) * FILM_SIZE[0] * FILM_DPI
        fig.text(x0 + 0.012, top - 0.090, value, color=color, fontsize=fitted(value, 15, room, bold=True), fontweight="bold")
        fig.text(x0 + 0.012, top - 0.135, sub, color=FILM_STYLE["muted"], fontsize=fitted(sub, 7.4, room))
    fair = caption.startswith("fair test:")  # the fair-test line of a film stands out in teal
    fig.text(0.035, 0.022, caption, color=FILM_STYLE["win"] if fair else FILM_STYLE["muted"], fontsize=8 if fair else 7.6,
             fontweight="bold" if fair else "normal")


def act_label(name):
    """'act II of V': the act's number among the acts after the title, in FILM_ACTS (§2.2)."""
    numerals = ["I", "II", "III", "IV", "V", "VI", "VII"]
    names = [act for act, _, _ in FILM_ACTS if act != "title"]
    return f"act {numerals[names.index(name)]} of {numerals[len(names) - 1]}"


def stage(fig, rect=None):
    """An empty axes for an act, without frame or ticks."""
    ax = fig.add_axes(rect or STAGE)
    ax.set_facecolor(FILM_STYLE["paper"])
    ax.set_axis_off()
    return ax


def plain_axes(ax):
    """The house axes style: light horizontal grid, no top and right spines."""
    ax.set_facecolor(FILM_STYLE["paper"])
    ax.grid(axis="y", color=FILM_STYLE["grid"], linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(FILM_STYLE["border"])
    ax.tick_params(colors=FILM_STYLE["muted"], labelsize=8, length=0)


def member_legend(ax, keys, y, row_step):
    """Colour squares with member labels in axes coordinates: five per line, four if there are more."""
    per_row = len(keys) if len(keys) <= 5 else 4
    step = 1 / per_row
    for i, key in enumerate(keys):
        row, col = divmod(i, per_row)
        x, line = col * step, y - row * row_step
        ax.add_patch(mpatches.Rectangle((x, line - row_step * 0.22), 0.016, row_step * 0.44, transform=ax.transAxes,
                                        color=FILM["member_color"][key], clip_on=False))
        ax.text(x + 0.024, line, FILM["member_label"][key], transform=ax.transAxes,
                fontsize=8, va="center", color=FILM_STYLE["ink"])


print(f"RENDER_FILMS = {RENDER_FILMS} · {int(FILM_SIZE[0] * FILM_DPI)}×{int(FILM_SIZE[1] * FILM_DPI)} · "
      f"output {FILM_DIR.relative_to(DATA_DIR.parent).as_posix()}/: {', '.join(FILMS_TO_RENDER)}")

### 1.2 The duel

The opponent of E1–E7 and the daily verdicts, from the common test hours of the inputs. A day counts as a tie when the two day MAEs differ by less than `TIE_MWH`.

In [ ]:
# Film VI, act IV: every full test day, ensemble against the best member
FILM["daily"]["verdict"] = np.select([FILM["daily"]["edge"] > TIE_MWH, FILM["daily"]["edge"] < -TIE_MWH], [1, -1], 0)

# E1–E7: the duel, on the common test hours, hour by hour and day by day
assert DUEL_AGAINST in ("smard", "best"), f"DUEL_AGAINST must be 'smard' or 'best', not {DUEL_AGAINST!r}"
best_label, best_color = FILM["best_label"], FILM["member_color"][FILM["best_member"]]
# SMARD: the palette's slate, always dashed
smard_color, SMARD_DASH = FILM["member_color"]["smard"], (0, (6, 3))
FILM["duel"] = {
    "against": DUEL_AGAINST,
    "label": "SMARD, the official" if DUEL_AGAINST == "smard" else f"{best_label}, the best member",
    "short": "official" if DUEL_AGAINST == "smard" else "best member",
    "color": smard_color if DUEL_AGAINST == "smard" else best_color,
    "other": "best" if DUEL_AGAINST == "smard" else "smard",
    "other_label": f"{best_label} (best member)" if DUEL_AGAINST == "smard" else "SMARD",
    "other_color": best_color if DUEL_AGAINST == "smard" else smard_color,
    "other_short": "best member" if DUEL_AGAINST == "smard" else "SMARD",
    "style": SMARD_DASH if DUEL_AGAINST == "smard" else "-",
    "other_style": (0, (2, 2)) if DUEL_AGAINST == "smard" else SMARD_DASH,
}
hourly = FILM["hourly"]
hourly["opp"] = hourly[DUEL_AGAINST]
hourly["other"] = hourly[FILM["duel"]["other"]]
hourly["gain"] = (hourly["opp"] - hourly["actual"]).abs() - (hourly["hero"] - hourly["actual"]).abs()  # > 0: ensemble better
in_full_days = hourly.index.normalize().isin(FILM["daily"].index)
by_day = hourly[in_full_days].groupby(hourly.index[in_full_days].normalize())
FILM["days"] = pd.DataFrame({
    "hero_mae": by_day.apply(lambda d: (d["hero"] - d["actual"]).abs().mean()),
    "opp_mae": by_day.apply(lambda d: (d["opp"] - d["actual"]).abs().mean()),
    "other_mae": by_day.apply(lambda d: (d["other"] - d["actual"]).abs().mean()),
    "opp_error": by_day.apply(lambda d: (d["opp"] - d["actual"]).mean()),  # signed, forecast − actual
    "change": by_day.apply(lambda d: (d["hero"] - d["opp"]).mean()),       # what the ensemble changes
    "spread": by_day["spread"].mean(),
    "percentile": by_day["percentile"].mean(),
    "regime": by_day["regime"].agg(lambda r: r.mode().iloc[0] if r.notna().any() else "–"),
})
FILM["days"]["edge"] = FILM["days"]["opp_mae"] - FILM["days"]["hero_mae"]  # > 0: ensemble better
FILM["days"]["verdict"] = np.select([FILM["days"]["edge"] > TIE_MWH, FILM["days"]["edge"] < -TIE_MWH], [1, -1], 0)

won, lost = int((FILM["days"]["verdict"] == 1).sum()), int((FILM["days"]["verdict"] == -1).sum())
print(f"duel: {FILM['hero_label']} against {FILM['duel']['label']} · {len(FILM['days'])} full test days, "
      f"won {won} : lost {lost} · mean daily edge {FILM['days']['edge'].mean():+,.0f} MWh")
display(FILM["monthly"].set_index(FILM["monthly"].index.strftime("%b %Y"))
        .style.format("{:+,.0f}", subset=["edge"]).format("{:,.0f}", subset=["hero", "best"])
        .set_caption("Film VI, act IV · test MAE per full month (MWh); edge = best member − ensemble"))

## <a id="sec-2"></a>2 Film VI · The second glass

### 2.1 The acts

Each act is a function `act_*(fig, t)` with `t` in `[0, 1]`, its own clock; the stagger constants
inside set when each element enters. Every frame is drawn from scratch (`fig.clf()`), so nothing can
pile up between frames.

In [ ]:
def act_title(fig, t):
    ax = stage(fig)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.text(0.5, 0.78, "GRIDSTRESS", ha="center", color=FILM_STYLE["kicker"], fontsize=13,
            fontweight="bold", alpha=ease(seg(t, 0.0, 0.25)))
    ax.text(0.5, 0.56, "THE SECOND GLASS", ha="center", color=FILM_STYLE["ink"], fontsize=40,
            fontweight="bold", alpha=ease(seg(t, 0.1, 0.45)))
    ax.text(0.5, 0.44, "Film VI · combining the regression models", ha="center", color=FILM_STYLE["muted"],
            fontsize=12, alpha=ease(seg(t, 0.3, 0.6)))
    fade = ease(seg(t, 0.5, 0.85))
    smard = " and SMARD's own forecast" if FILM["smard_member"] else ""
    ax.text(0.5, 0.24, f"{FILM['n_registry']} models{smard}: one forecast per hour, mixed.", ha="center",
            color=FILM_STYLE["muted"], fontsize=11, alpha=fade)
    ax.text(0.5, 0.16, "Does the mix beat the best single model, on a year it never saw?", ha="center",
            color=FILM_STYLE["ink"], fontsize=11, fontweight="bold", alpha=fade)
    (v0, v1), (t0, t1) = FILM["validation"], FILM["test"]
    chrome(
        fig, "", "Many models, one forecast per hour", "",
        [
            ("MEMBERS", f"{FILM['n_registry']}" + (" + SMARD" if FILM["smard_member"] else ""), "the forecasts being combined", FILM_STYLE["ink"]),
            ("VALIDATION", f"{v0:%d %b %y} – {v1:%d %b %y}", "every weight is fitted here", FILM_STYLE["ink"]),
            ("TEST", f"{t0:%d %b %y} – {t1:%d %b %y}", "scored once, never fitted on", FILM_STYLE["ink"]),
            ("ISSUE TIME", FILM["issue"], "every forecast is made before the day", FILM_STYLE["ink"]),
        ],
        "residual load, day-ahead · MWh per hour · every number on screen is read from this run of ensemble-claude.ipynb",
    )


def act_field(fig, t):
    field = FILM["field"]
    keys = list(field.index)
    members = FILM["members"]
    number = {key: i + 1 for i, key in enumerate(members)}
    ax = fig.add_axes([0.20, 0.12, 0.30, 0.66])
    plain_axes(ax)
    ax.grid(axis="x", color=FILM_STYLE["grid"], linewidth=0.8)
    ax.grid(axis="y", visible=False)
    scale_keys = [key for key in keys if key in members or key == "smard"]
    top = float(field.loc[scale_keys, "MAE"].max()) * 1.45  # far-off bars are clipped, with their value
    ax.set_xlim(0, top)
    ax.set_ylim(len(keys) - 0.4, -0.6)
    ax.set_yticks([])
    ax.xaxis.set_major_formatter(lambda value, _: f"{value:,.0f}")
    ax.set_xlabel("validation MAE (MWh)", color=FILM_STYLE["muted"], fontsize=8.5)
    for i, key in enumerate(keys):
        grow = ease(seg(t, 0.04 + 0.04 * i, 0.30 + 0.04 * i))
        mae = float(field.at[key, "MAE"])
        is_best = key == FILM["best_member"]
        clipped = mae > top * 0.88
        ax.barh(i, min(mae, top * 0.88) * grow, height=0.66, color=FILM["member_color"][key],
                alpha=0.95 if key in members else 0.45)
        prefix = f"{number[key]} · " if key in number else ""
        ax.text(-top * 0.02, i, prefix + FILM["member_label"][key], ha="right", va="center", fontsize=8.5,
                color=FILM_STYLE["ink"], fontweight="bold" if is_best else "normal", clip_on=False)
        if grow >= 1:
            ax.text(min(mae, top * 0.88) + top * 0.012, i, f"{mae:,.0f}" + (" ▸" if clipped else ""), va="center",
                    fontsize=7.6, color=FILM_STYLE["muted"])
    wall = ease(seg(t, 0.42, 0.55))
    smard_mae = float(field.at["smard", "MAE"])
    if wall > 0:
        ax.plot([smard_mae, smard_mae], [-0.6, -0.6 + wall * (len(keys) + 0.2)], color=FILM["member_color"]["smard"],
                linewidth=1.4, linestyle=(0, (5, 4)))
    if wall >= 1:
        ax.text(smard_mae, -0.75, "SMARD · the wall", ha="center", fontsize=8, color=FILM["member_color"]["smard"],
                fontweight="bold", clip_on=False)
    best_in = ease(seg(t, 0.58, 0.70))
    if best_in > 0:
        i = keys.index(FILM["best_member"])
        ax.text(float(field.at[FILM["best_member"], "MAE"]) + top * 0.16, i, "◀ best single member",
                va="center", fontsize=8.5, color=FILM_STYLE["ink"], fontweight="bold", alpha=best_in)

    heat_in = ease(seg(t, 0.70, 0.86))
    if heat_in > 0:
        hx = fig.add_axes([0.565, 0.30, 0.165, 0.293])
        corr = FILM["correlation"].to_numpy()
        hx.imshow(corr, cmap="Blues", vmin=0.5, vmax=1, alpha=heat_in)
        hx.set_xticks(range(len(members)), [str(number[k]) for k in members], fontsize=7.5)
        hx.set_yticks(range(len(members)), [str(number[k]) for k in members], fontsize=7.5)
        hx.tick_params(length=0, colors=FILM_STYLE["muted"])
        for side in hx.spines.values():
            side.set_visible(False)
        if len(members) <= 8:
            for r in range(len(members)):
                for c in range(len(members)):
                    hx.text(c, r, f"{corr[r, c]:.2f}", ha="center", va="center", fontsize=6.2,
                            color="white" if corr[r, c] > 0.85 else FILM_STYLE["ink"], alpha=heat_in)
        hx.set_title("error correlation, members", fontsize=8.5, color=FILM_STYLE["ink"], loc="left")

    best = FILM["best_member"]
    chrome(
        fig, act_label("field"), "The field: who is there to combine",
        f"validation year · {FILM['split']} split · {FILM['field_hours']:,} common hours",
        [
            ("ENTRANTS", f"{len(keys)}", f"{FILM['n_registry']} models · SMARD · seasonal naive", FILM_STYLE["ink"]),
            ("THE WALL", f"{smard_mae:,.0f} MWh", "SMARD's validation MAE", FILM["member_color"]["smard"]),
            ("BEST SINGLE MEMBER", f"{float(field.at[best, 'MAE']):,.0f} MWh", FILM["member_label"][best], FILM["member_color"][best]),
            ("ERRORS AGREE", f"r = {FILM['mean_corr']:.2f}", "mean correlation between the models", FILM_STYLE["ink"]),
        ],
        "bars = validation MAE · faint = not a member · ▸ = bar cut at the frame · numbers 1… link the bars to the "
        "correlation matrix · a mix only gains where the errors differ",
    )


def act_pour(fig, t):
    regimes = FILM["regimes"]
    glasses = [(r["name"], r["weights"], f"{r['hours']:,} h · own {r['own_share']:.0%}") for r in regimes]
    glasses.append(("global", FILM["global_weights"], "all fit hours"))
    members = FILM["members"]
    ax = stage(fig, [0.045, 0.08, 0.68, 0.70])
    ax.set_xlim(0, len(glasses))
    ax.set_ylim(-0.30, 1.22)
    width, taper = 0.58, 0.05
    fills = [ease(seg(t, 0.05 + i * 0.12, 0.25 + i * 0.12)) for i in range(len(glasses))]
    for i, ((name, weights, sub), fill) in enumerate(zip(glasses, fills)):
        center = i + 0.5
        left, right = center - width / 2, center + width / 2
        ax.plot([left, left + taper, right - taper, right], [1.0, 0.0, 0.0, 1.0],
                color=FILM_STYLE["muted"], linewidth=1.8, solid_capstyle="round")
        level = 0.0
        for key, weight in zip(members, weights):
            height = max(0.0, min(fill - level, weight))
            if height > 0:
                inset0, inset1 = taper * (1 - level), taper * (1 - level - height)
                ax.fill([left + inset0, left + inset1, right - inset1, right - inset0],
                        [level, level + height, level + height, level],
                        color=FILM["member_color"][key], alpha=0.92, linewidth=0)
                if height >= 0.07 and weight >= 0.07:
                    ax.text(center, level + height / 2, f"{weight:.0%}", ha="center", va="center",
                            fontsize=7.6, color="white", fontweight="bold")
            level += weight
        if 0 < fill < 1:
            ax.plot([center, center], [1.15, fill + 0.02], color=FILM_STYLE["faint"], linewidth=3, alpha=0.8)
        ax.text(center, -0.09, name, ha="center", fontsize=9, fontweight="bold",
                color=FILM_STYLE["ink"] if name != "global" else FILM_STYLE["kicker2"])
        ax.text(center, -0.18, sub, ha="center", fontsize=7.4, color=FILM_STYLE["muted"])
    member_legend(ax, members, y=0.975, row_step=0.045)

    current = max(0, sum(f > 0 for f in fills) - 1)
    name, weights, _ = glasses[current]
    top = int(np.argmax(weights))
    if name == "global":
        regime_text, regime_sub = "global", "the weights of the weighted method"
    else:
        entry = regimes[current]
        edges = FILM["regime_edges"]
        if FILM["regime_by_level"]:
            low = "−∞" if current == 0 else f"{edges[current - 1]:,.0f}"
            high = f"{edges[current]:,.0f}" if current < len(edges) else "∞"
            regime_sub = f"members' mean {low} .. {high} MWh"
        else:
            regime_sub = f"{entry['hours']:,} fit hours"
        regime_text = name
    chrome(
        fig, act_label("pour"), "The pour: who gets into the glass, regime by regime",
        f"weights ≥ 0, summing to 1 · fitted on the validation year · {FILM['split']} split",
        [
            ("FILLING", regime_text, regime_sub, FILM_STYLE["ink"]),
            ("FIT HOURS", "all" if name == "global" else f"{regimes[current]['hours']:,}",
             "the LP's hours" if name == "global" else f"own share {regimes[current]['own_share']:.0%}", FILM_STYLE["ink"]),
            ("TOP MEMBER", f"{weights[top]:.0%}", FILM["member_label"][members[top]], FILM["member_color"][members[top]]),
            ("SHRINKAGE", f"k = {FILM['shrink_hours']}", "small regimes lean on the global glass", FILM_STYLE["ink"]),
        ],
        "one glass per regime of regime_weighted · own share = n / (n + k) of the regime's own LP, "
        "the rest from the global weights (last glass)",
    )


def act_adaptive(fig, t):
    entry = FILM["adaptive"]
    weights = entry["weights"]
    members = FILM["members"]
    ax = fig.add_axes([0.07, 0.12, 0.655, 0.58])
    plain_axes(ax)
    shown = max(2, int(len(weights) * ease(seg(t, 0.03, 0.85))))
    part = weights.iloc[:shown]
    ax.stackplot(part.index, part.T.to_numpy(), colors=[FILM["member_color"][k] for k in members], alpha=0.88)
    ax.set_xlim(weights.index[0], weights.index[-1])
    ax.set_ylim(0, 1)
    ax.yaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
    ax.set_ylabel("weight", color=FILM_STYLE["muted"], fontsize=8.5)
    ax.xaxis.set_major_locator(mdates.MonthLocator((1, 4, 7, 10)))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %y"))
    test_start = FILM["test"][0]
    ax.axvline(test_start, color=FILM_STYLE["ink"], linewidth=1.2, linestyle=(0, (5, 4)))
    ax.text(test_start, 1.02, " test year →", fontsize=8.5, color=FILM_STYLE["ink"], fontweight="bold",
            transform=ax.get_xaxis_transform())
    ax.text(weights.index[0], 1.02, "validation year", fontsize=8.5, color=FILM_STYLE["muted"],
            transform=ax.get_xaxis_transform())
    legend = stage(fig, [0.07, 0.745, 0.655, 0.065])
    legend.set_xlim(0, 1)
    legend.set_ylim(0, 1)
    member_legend(legend, members, y=0.75, row_step=0.5)

    today = part.index[-1]
    row = part.iloc[-1]
    top = row.idxmax()
    candidates = " · ".join(f"{name.replace(' days', ' d')} {mae:,.0f}" for name, mae in entry["candidates"].items()) + " MWh"
    chrome(
        fig, act_label("adaptive"), f"The adaptive glass: weights that follow the last {entry['window']}",
        "weight ∝ 1 / MAE over the trailing window · only errors known at each day's cutoff",
        [
            ("WINDOW", entry["window"], f"picked on validation · {candidates}", FILM_STYLE["ink"]),
            ("DELIVERY DAY", f"{today:%d %b %Y}", "test year" if today >= test_start else "validation year", FILM_STYLE["ink"]),
            ("LARGEST WEIGHT", f"{row[top]:.0%}", FILM["member_label"][top], FILM["member_color"][top]),
            ("EQUAL SHARE", f"{1 / len(members):.0%}", "1 / number of members", FILM_STYLE["muted"]),
        ],
        "one band per member, stacked to 1 · the first validation days have no forecast: their window "
        "would reach before the saved forecasts",
    )


def act_year(fig, t):
    monthly, daily = FILM["monthly"], FILM["daily"]
    ax = fig.add_axes([0.07, 0.47, 0.655, 0.32])
    plain_axes(ax)
    n = len(monthly)
    edges = monthly["edge"].to_numpy()
    span = max(abs(edges).max(), TIE_MWH) * 1.35
    ax.set_xlim(-0.6, n - 0.4)
    ax.set_ylim(min(-span * 0.35, edges.min() * 1.35), span)
    ax.axhline(0, color=FILM_STYLE["ink"], linewidth=1)
    ax.set_xticks(range(n), monthly.index.strftime("%b\n%y"), fontsize=7.5)
    ax.yaxis.set_major_formatter(lambda value, _: signed(value))
    ax.set_ylabel("MAE edge (MWh)", color=FILM_STYLE["muted"], fontsize=8.5)
    shown = -1
    for i, edge in enumerate(edges):
        grow = ease(seg(t, 0.04 + i * 0.045, 0.14 + i * 0.045))
        if grow > 0:
            shown = i
        color = FILM_STYLE["win"] if edge >= 0 else FILM_STYLE["loss"]
        ax.bar(i, edge * grow, width=0.62, color=color)
        if grow >= 1:
            ax.text(i, edge + (span * 0.04 if edge >= 0 else -span * 0.04), signed(round(edge)), ha="center",
                    va="bottom" if edge >= 0 else "top", fontsize=7.4, color=color, fontweight="bold")

    cal = fig.add_axes([0.07, 0.10, 0.655, 0.27])
    cal.set_axis_off()
    first = daily.index[0]
    week = ((daily.index - (first - pd.Timedelta(days=first.dayofweek))).days // 7).to_numpy()
    weeks = int(week.max()) + 1
    cal.set_xlim(-2.2, weeks + 0.2)
    cal.set_ylim(-1.6, 7.2)
    colors = {1: FILM_STYLE["win"], 0: FILM_STYLE["tie"], -1: FILM_STYLE["loss"]}
    filled = int(len(daily) * seg(t, 0.30, 0.95))
    for k in range(filled):
        stamp = daily.index[k]
        cal.add_patch(mpatches.Rectangle((week[k], 6 - stamp.dayofweek), 0.88, 0.88, color=colors[int(daily["verdict"].iloc[k])], linewidth=0))
    for d, name in enumerate(DAY_NAMES):
        cal.text(-0.6, 6 - d + 0.44, name, ha="right", va="center", fontsize=6.8, color=FILM_STYLE["muted"])
    for stamp in daily.index[daily.index.day == 1]:
        k = daily.index.get_loc(stamp)
        cal.text(week[k], -0.75, f"{stamp:%b}", fontsize=6.8, color=FILM_STYLE["muted"])
    cal.text(-2.2, 7.25, "every test day, judged: ensemble day MAE against the best member's", fontsize=8.5,
             color=FILM_STYLE["ink"], fontweight="bold")

    judged = daily.iloc[:filled]
    won, lost, tied = int((judged["verdict"] == 1).sum()), int((judged["verdict"] == -1).sum()), int((judged["verdict"] == 0).sum())
    ahead = int((edges[: shown + 1] > 0).sum()) if shown >= 0 else 0
    month = monthly.index[max(shown, 0)]
    edge = edges[max(shown, 0)]
    chrome(
        fig, act_label("year"), "The test year, judged month by month",
        f"{FILM['hero_label']} against {FILM['best_label']}, the best member on validation · above 0 = ensemble better",
        [
            ("MONTH", month.strftime("%b %Y"), "full calendar months of the test year", FILM_STYLE["ink"]),
            ("EDGE THIS MONTH", signed(edge, unit="MWh"), "best member MAE − ensemble MAE",
             FILM_STYLE["win"] if edge >= 0 else FILM_STYLE["loss"]),
            ("MONTHS AHEAD", f"{ahead} of {shown + 1}", f"of {n} full months", FILM_STYLE["ink"]),
            ("DAYS WON", f"{won} : {lost}", f"ensemble : best member · {tied} ties (±{TIE_MWH:,.0f} MWh)", FILM_STYLE["ink"]),
        ],
        "teal = ensemble better · coral = best member better · cream = tie · common test hours of §5 · "
        "days with at least 23 common hours",
    )


def act_verdict(fig, t):
    verdict = FILM["verdict"].sort_values("MAE")
    hero = FILM["hero"]
    ax = fig.add_axes([0.25, 0.44, 0.40, 0.36])
    plain_axes(ax)
    ax.grid(axis="x", color=FILM_STYLE["grid"], linewidth=0.8)
    ax.grid(axis="y", visible=False)
    low, high = verdict["MAE"].min(), verdict["MAE"].max()
    pad = (high - low) * 0.15 + 1
    ax.set_xlim(low - pad, high + pad)
    ax.set_ylim(len(verdict) - 0.5, -0.7)
    ax.set_yticks([])
    ax.xaxis.set_major_formatter(lambda value, _: f"{value:,.0f}")
    ax.set_xlabel("test MAE (MWh), common hours", color=FILM_STYLE["muted"], fontsize=8.5)
    best_mae = float(verdict.loc[verdict["kind"] == "best", "MAE"].iloc[0])
    ax.axvline(best_mae, color=FILM["member_color"][FILM["best_member"]], linewidth=1.2, linestyle=(0, (5, 4)))
    for i, (_, row) in enumerate(verdict.iterrows()):
        slide = ease(seg(t, 0.05 + 0.08 * i, 0.25 + 0.08 * i))
        x = high + pad - (high + pad - row["MAE"]) * slide
        ax.plot([x], [i], "o", markersize=9, color=row["color"], markeredgecolor="white", zorder=3)
        name = row["label"] + ("  (best member)" if row["kind"] == "best" else "")
        ax.text(low - pad - (high - low) * 0.03, i, name, ha="right", va="center", fontsize=8.5, clip_on=False,
                color=FILM_STYLE["ink"], fontweight="bold" if row["label"] == FILM["hero_label"] else "normal")
        if slide >= 1:
            note = f"{row['MAE']:,.0f}" + (f" · {signed(row['skill_vs_best'], 1)} %" if row["kind"] == "ensemble" else "")
            ax.text(row["MAE"], i - 0.32, note, ha="center", fontsize=7.4, color=row["color"], fontweight="bold")

    strip_in = ease(seg(t, 0.55, 0.75))
    if strip_in > 0:
        strip = stage(fig, [0.045, 0.115, 0.68, 0.17])
        strip.set_xlim(0, 5)
        strip.set_ylim(0, 1)
        boxes = [(name.replace("_", " "), row["skill_vs_best"], f"{row['hours']:,.0f} h")
                 for name, row in FILM["tails"].iterrows()]
        for i, (name, skill, sub) in enumerate(boxes):
            color = FILM_STYLE["muted"] if pd.isna(skill) else FILM_STYLE["win"] if skill >= 0 else FILM_STYLE["loss"]
            strip.add_patch(mpatches.Rectangle((i + 0.04, 0.05), 0.92, 0.85, facecolor=color, alpha=0.10 * strip_in,
                                               edgecolor=color, linewidth=1))
            strip.text(i + 0.12, 0.70, name, fontsize=8.5, fontweight="bold", color=FILM_STYLE["ink"], alpha=strip_in)
            strip.text(i + 0.12, 0.38, signed(skill, 1) + ("" if pd.isna(skill) else " %"), fontsize=13, fontweight="bold", color=color, alpha=strip_in)
            strip.text(i + 0.12, 0.15, f"vs best member · {sub}", fontsize=6.6, color=FILM_STYLE["muted"], alpha=strip_in)
        gap = hero["coverage"] - hero["nominal"]
        color = FILM_STYLE["win"] if abs(gap) <= 1 else FILM_STYLE["loss"]
        strip.add_patch(mpatches.Rectangle((4.04, 0.05), 0.92, 0.85, facecolor=color, alpha=0.10 * strip_in,
                                           edgecolor=color, linewidth=1))
        strip.text(4.12, 0.70, f"{hero['nominal']:.0f} % band", fontsize=8.5, fontweight="bold", color=FILM_STYLE["ink"], alpha=strip_in)
        strip.text(4.12, 0.38, f"{hero['coverage']:.1f} %", fontsize=13, fontweight="bold", color=color, alpha=strip_in)
        strip.text(4.12, 0.15, f"coverage · {signed(gap, 1)} pp", fontsize=6.8, color=FILM_STYLE["muted"], alpha=strip_in)

    last = ease(seg(t, 0.82, 0.95))
    if last > 0:
        better = hero["skill_vs_best"] > 0
        line = (f"{FILM['hero_label']} {'beats' if better else 'does not beat'} the best single member: "
                f"{signed(hero['skill_vs_best'], 1)} %, ahead in {hero['months_beating_best']} of {hero['months']} months")
        fig.text(0.39, 0.335, line, ha="center", fontsize=11.5, fontweight="bold", alpha=last,
                 color=FILM["hero_color"] if better else FILM_STYLE["loss"])
    chrome(
        fig, act_label("verdict"), "The verdict, on the year it never saw",
        f"test year · {hero['hours']:,} common hours · scored once (§5) · {FILM['split']} split",
        [
            ("ENSEMBLE MAE", f"{hero['MAE']:,.0f} MWh", FILM["hero_label"], FILM["hero_color"]),
            ("VS BEST MEMBER", f"{signed(hero['skill_vs_best'], 1)} %", f"{FILM['best_label']} · {hero['months_beating_best']} of {hero['months']} months",
             FILM_STYLE["win"] if hero["skill_vs_best"] > 0 else FILM_STYLE["loss"]),
            ("VS SMARD", f"{signed(hero['skill_vs_smard'], 1)} %", "skill on the same hours",
             FILM_STYLE["win"] if hero["skill_vs_smard"] > 0 else FILM_STYLE["loss"]),
            ("BAND WIDTH", f"{hero['width']:,.0f} MWh", f"mean width of the {hero['nominal']:.0f} % band", FILM_STYLE["ink"]),
        ],
        f"dots = test MAE · dashed line = best member · % = skill vs best member · tails binned by the actual · "
        f"ensemble shown: {FILM['hero_pick']}",
    )

### 2.2 Timeline and key frames

A contact sheet with one late frame per act always renders, so the film can be checked without
rendering it.

In [ ]:
FILM_ACTS = [("title", act_title, 5), ("field", act_field, 14), ("pour", act_pour, 16)]
if FILM["adaptive"] is not None:
    FILM_ACTS.append(("adaptive", act_adaptive, 12))
FILM_ACTS += [("year", act_year, 16), ("verdict", act_verdict, 13)]
FILM_SECONDS = sum(seconds for _, _, seconds in FILM_ACTS)


def draw_at(fig, second):
    """Draw the frame at `second` of the film."""
    fig.clf()
    start = 0
    for _, render, seconds in FILM_ACTS:
        if second < start + seconds or start + seconds >= FILM_SECONDS:
            render(fig, (second - start) / seconds)
            return
        start += seconds


def film_frame(second, dpi=60):
    """One frame as an image array."""
    fig = plt.figure(figsize=FILM_SIZE, dpi=dpi)
    draw_at(fig, second)
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=dpi)
    plt.close(fig)
    buffer.seek(0)
    return plt.imread(buffer)


KEY_SECONDS = list(np.cumsum([0] + [s for _, _, s in FILM_ACTS])[:-1] + 0.93 * np.array([s for _, _, s in FILM_ACTS]))
sheet_rows = int(np.ceil(len(KEY_SECONDS) / 2))
fig, axes = plt.subplots(sheet_rows, 2, figsize=(16, 4.6 * sheet_rows))
for ax in axes.ravel():
    ax.set_axis_off()
for ax, second in zip(axes.ravel(), KEY_SECONDS):
    ax.imshow(film_frame(second))
fig.tight_layout(pad=0.4)
plt.show()
print(f"{len(FILM_ACTS)} acts, {FILM_SECONDS} s: " + " · ".join(f"{name} {seconds} s" for name, _, seconds in FILM_ACTS))

## <a id="sec-3"></a>3 The companion films E1–E7

Each film is a function `film_*(fig, k)` that draws frame `k` from scratch. Shared rules: the
common test hours of §5; ink = actual, the opponent in its colour, the ensemble in its colour, the
other reference as a thin line; teal = ensemble better, coral = opponent better, cream = a tie within
`±TIE_MWH`. Every
film holds its last frame for `FILM_HOLD` seconds.

In [ ]:
import textwrap
from matplotlib.colors import LinearSegmentedColormap

FILM_HOLD = 2  # seconds the last frame of E1–E7 stays on screen
# The house level colours of EDA-hari.ipynb (green low → red high), without its cream middle, which
# would vanish on the white frame
LEVEL_SCALE = LinearSegmentedColormap.from_list(  # low → high load, the palette's blue → amber → vermillion
    "gridstress_level", ["#2E64B0", "#7385CB", "#E7A118", "#C8553D"])
VERDICT_COLOR = {1: FILM_STYLE["win"], 0: FILM_STYLE["tie"], -1: FILM_STYLE["loss"]}
E_DAYS = list(FILM["days"].index)
E_HOURS_OF = {day: FILM["hourly"].loc[FILM["hourly"].index.normalize() == day] for day in E_DAYS}
DUEL = FILM["duel"]


def mwh_axis(ax, which="y"):
    """Thousands separators on an MWh axis."""
    axis = ax.yaxis if which == "y" else ax.xaxis
    axis.set_major_formatter(lambda value, _: f"{value:,.0f}")


def forecast_lines(ax, part, x, widths=(2.2, 1.0, 1.8, 2.0)):
    """Actual, the other reference (thin), the opponent and the ensemble of `part` against `x`."""
    ax.plot(x, part["actual"], color=FILM["member_color"]["actual"], linewidth=widths[0])
    ax.plot(x, part["other"], color=DUEL["other_color"], linewidth=widths[1], alpha=0.55, linestyle=DUEL["other_style"])
    ax.plot(x, part["opp"], color=DUEL["color"], linewidth=widths[2], linestyle=DUEL["style"])
    ax.plot(x, part["hero"], color=FILM["hero_color"], linewidth=widths[3])


def line_caption():
    return (f"ink = actual · {DUEL['label']} (its colour) · {FILM['hero_label']} = ensemble (its colour) · "
            f"thin dashed = {DUEL['other_label']}")


# ---------- E1 · the duel: one frame per full day of E1_WINDOW ----------
E1_FIRST, E1_LAST = (pd.Timestamp(day) for day in E1_WINDOW) if E1_WINDOW else (E_DAYS[0], E_DAYS[-1])
E1_DAYS = [day for day in E_DAYS if E1_FIRST <= day <= E1_LAST]
if not E1_DAYS:
    raise ValueError(f"E1_WINDOW {E1_WINDOW} holds no test day; the test days run {E_DAYS[0]:%d %b %Y} .. {E_DAYS[-1]:%d %b %Y}")
if E1_DAYS[0] > E1_FIRST or E1_DAYS[-1] < E1_LAST:
    print(f"E1: the window asks for {E1_FIRST:%d %b %Y} .. {E1_LAST:%d %b %Y}; the film inputs hold test days "
          f"{E_DAYS[0]:%d %b %Y} .. {E_DAYS[-1]:%d %b %Y}, so E1 runs {E1_DAYS[0]:%d %b %Y} .. {E1_DAYS[-1]:%d %b %Y}")
E1_TABLE = FILM["days"].loc[E1_DAYS]
E1_HOURS = FILM["hourly"].loc[FILM["hourly"].index.normalize().isin(E1_DAYS)]
E1_LIMITS = (float(E1_HOURS[["actual", "hero", "opp", "other"]].min().min()),
             float(E1_HOURS[["actual", "hero", "opp", "other"]].max().max()))
E1_GAIN = float(E1_HOURS["gain"].abs().quantile(0.995))
E1_RACE = pd.DataFrame({column: E1_TABLE[f"{column}_mae"].expanding().mean() for column in ("hero", "opp", "other")})
E1_MONDAY = E1_DAYS[0] - pd.Timedelta(days=E1_DAYS[0].dayofweek)
E1_WEEK = ((pd.DatetimeIndex(E1_DAYS) - E1_MONDAY).days // 7).to_numpy()
E1_LINES = [  # what each line is, in the legend and in the race
    ("actual", "Actual residual load", FILM["member_color"]["actual"], "-", 2.4),
    ("opp", DUEL["label"].replace(", the official", " · official forecast").replace(", the best member", " · best member"),
     DUEL["color"], DUEL["style"], 1.9),
    ("hero", f"Ensemble · {FILM['hero_label']}", FILM["hero_color"], "-", 2.2),
    ("other", DUEL["other_label"], DUEL["other_color"], DUEL["other_style"], 1.1),
]


def legend_row(fig, x0, y, entries, width=0.165):
    """Line swatches with their labels across the figure, in figure coordinates."""
    for i, (label, color, style, line_width) in enumerate(entries):
        x = x0 + i * width
        fig.add_artist(plt.Line2D([x, x + 0.022], [y, y], transform=fig.transFigure, color=color,
                                  linestyle=style, linewidth=max(line_width, 1.6)))
        fig.text(x + 0.028, y, label, fontsize=fitted(label, 8.2, (width - 0.034) * FILM_SIZE[0] * FILM_DPI),
                 va="center", color=FILM_STYLE["ink"])


def film_e1(fig, position):
    """The frame at `position` days into E1_DAYS: day int(position) is being drawn, the week glides with it."""
    fig.clf()
    fig.patch.set_facecolor(FILM_STYLE["paper"])
    i = min(int(position), len(E1_DAYS) - 1)
    day = E1_DAYS[i]
    shown_until = day + pd.Timedelta(days=min(position - i, 1.0) if position < len(E1_DAYS) else 1.0)
    window_start = shown_until - pd.Timedelta(days=7)
    window = E1_HOURS.loc[(E1_HOURS.index >= window_start) & (E1_HOURS.index < shown_until)]
    if len(window) < 2:
        window = E1_HOURS.iloc[:2]
    pad = (E1_LIMITS[1] - E1_LIMITS[0]) * 0.05

    legend_row(fig, 0.07, 0.815, [(label, color, style, width) for _, label, color, style, width in E1_LINES])

    top = fig.add_axes([0.07, 0.52, 0.655, 0.27])
    plain_axes(top)
    for column, _, color, style, width in reversed(E1_LINES):
        top.plot(window.index, window[column], color=color, linestyle=style, linewidth=width,
                 alpha=0.6 if column == "other" else 1)
    top.set_ylim(E1_LIMITS[0] - pad, E1_LIMITS[1] + pad)
    top.set_xlim(max(window_start, E1_DAYS[0]), max(shown_until, E1_DAYS[0] + pd.Timedelta(days=7)))
    top.tick_params(labelbottom=False)
    top.set_ylabel("MWh per hour", color=FILM_STYLE["muted"], fontsize=8.5)
    top.text(0.995, 0.97, "the last seven days, hour by hour", transform=top.transAxes, ha="right", va="top",
             fontsize=8, color=FILM_STYLE["muted"])
    mwh_axis(top)

    gain = fig.add_axes([0.07, 0.385, 0.655, 0.115], sharex=top)
    plain_axes(gain)
    gain.bar(window.index, window["gain"], width=1 / 24 * 0.9,
             color=np.where(window["gain"] >= 0, FILM_STYLE["win"], FILM_STYLE["loss"]))
    gain.axhline(0, color=FILM_STYLE["ink"], linewidth=0.8)
    gain.set_ylim(-E1_GAIN, E1_GAIN)
    gain.set_ylabel("MWh", color=FILM_STYLE["muted"], fontsize=7.5)
    gain.text(0.005, 0.93, "▲ ensemble closer to the actual this hour", transform=gain.transAxes, fontsize=7.6,
              color=FILM_STYLE["win"], fontweight="bold", va="top")
    gain.text(0.005, 0.07, f"▼ {DUEL['short']} closer", transform=gain.transAxes, fontsize=7.6,
              color=FILM_STYLE["loss"], fontweight="bold", va="bottom")
    gain.xaxis.set_major_locator(mdates.DayLocator())
    gain.xaxis.set_major_formatter(mdates.DateFormatter("%a %d %b"))
    mwh_axis(gain)

    race = fig.add_axes([0.07, 0.075, 0.34, 0.21])
    plain_axes(race)
    shown = E1_RACE.iloc[: i + 1]
    names = {"hero": "ensemble", "opp": DUEL["short"], "other": DUEL["other_short"]}
    for column, _, color, style, width in E1_LINES[1:]:
        race.plot(shown.index, shown[column], color=color, linewidth=2.0 if column != "other" else 1.2, linestyle=style)
        race.annotate(f"{names[column]} {shown[column].iloc[-1]:,.0f}", (shown.index[-1], shown[column].iloc[-1]),
                      xytext=(5, 0), textcoords="offset points", fontsize=7.4, color=color, fontweight="bold", va="center")
    settled = E1_RACE.iloc[min(len(E1_RACE) - 1, 13):]
    race.set_xlim(E1_DAYS[0], E1_DAYS[-1] + (E1_DAYS[-1] - E1_DAYS[0]) * 0.32)
    race.set_ylim(settled.min().min() * 0.92, settled.max().max() * 1.08)
    race.xaxis.set_major_locator(mdates.MonthLocator((1, 4, 7, 10)))
    race.xaxis.set_major_formatter(mdates.DateFormatter("%b %y"))
    race.set_title("the race · average error so far (MAE, MWh per hour) · lower is better", loc="left",
                   fontsize=8.2, color=FILM_STYLE["ink"])
    mwh_axis(race)

    wall = fig.add_axes([0.465, 0.105, 0.26, 0.18])
    weeks = int(E1_WEEK.max()) + 1
    for j, d in enumerate(E1_DAYS):  # one square per day, a small gap between them; empty until it is judged
        x, y = E1_WEEK[j], 6 - d.dayofweek
        if j <= i:
            wall.add_patch(mpatches.Rectangle((x + 0.1, y + 0.1), 0.8, 0.8, linewidth=0,
                                              facecolor=VERDICT_COLOR[int(E1_TABLE["verdict"].iloc[j])]))
        else:
            wall.add_patch(mpatches.Rectangle((x + 0.1, y + 0.1), 0.8, 0.8, linewidth=0.5, facecolor="none",
                                              edgecolor=FILM_STYLE["grid"]))
    wall.add_patch(mpatches.Rectangle((E1_WEEK[i] + 0.02, 6 - E1_DAYS[i].dayofweek + 0.02), 0.96, 0.96, linewidth=1.4,
                                      facecolor="none", edgecolor=FILM_STYLE["ink"]))  # today
    wall.set_xlim(0, weeks)
    wall.set_ylim(0, 7)
    wall.set_yticks([6.5 - n for n in range(7)], DAY_NAMES, fontsize=6.2)
    firsts = [(E1_WEEK[j], d) for j, d in enumerate(E1_DAYS) if d.day == 1]
    wall.set_xticks([w + 0.5 for w, _ in firsts], [f"{d:%b}" for _, d in firsts], fontsize=6.2)
    wall.tick_params(length=0, colors=FILM_STYLE["muted"])
    for side in wall.spines.values():
        side.set_visible(False)
    wall.set_title("every day, judged: who had the lower error that day", loc="left", fontsize=8.2, color=FILM_STYLE["ink"])
    for n, (label, color) in enumerate([("ensemble day", FILM_STYLE["win"]), (f"{DUEL['short']} day", FILM_STYLE["loss"]),
                                        (f"tie (within ±{TIE_MWH:,.0f} MWh)", FILM_STYLE["tie"])]):
        x = 0.465 + n * 0.082
        fig.patches.append(mpatches.Rectangle((x, 0.048), 0.011, 0.02, transform=fig.transFigure, facecolor=color,
                                              edgecolor=FILM_STYLE["border"], linewidth=0.6))
        fig.text(x + 0.015, 0.058, label, fontsize=7.2, va="center", color=FILM_STYLE["ink"])

    days = E1_TABLE.iloc[: i + 1]
    today = E1_TABLE.iloc[i]
    won, lost, tied = int((days["verdict"] == 1).sum()), int((days["verdict"] == -1).sum()), int((days["verdict"] == 0).sum())
    edge = E1_RACE["opp"].iloc[i] - E1_RACE["hero"].iloc[i]
    chrome(
        fig, f"day {i + 1} of {len(E1_DAYS)}", f"{day:%A %d %B %Y}",
        f"days won: ensemble {won} · {DUEL['short']} {lost} · ties {tied}   |   ensemble ahead by "
        f"{edge:,.0f} MWh per hour so far" if edge >= 0 else
        f"days won: ensemble {won} · {DUEL['short']} {lost} · ties {tied}   |   {DUEL['short']} ahead by "
        f"{-edge:,.0f} MWh per hour so far",
        [
            ("ACTUAL TODAY", f"{E_HOURS_OF[day]['actual'].mean():,.0f} MWh", "residual load, mean per hour", FILM["member_color"]["actual"]),
            ("ENSEMBLE ERROR TODAY", f"{today['hero_mae']:,.0f} MWh", "mean absolute error per hour", FILM["hero_color"]),
            (f"{DUEL['short'].upper()} ERROR TODAY", f"{today['opp_mae']:,.0f} MWh", "mean absolute error per hour", DUEL["color"]),
            ("TODAY'S WINNER", {1: "ensemble", 0: "tie", -1: DUEL["short"]}[int(today["verdict"])],
             f"by {abs(today['edge']):,.0f} MWh per hour", VERDICT_COLOR[int(today["verdict"])]
             if today["verdict"] != 0 else FILM_STYLE["muted"]),
        ],
        E1_FOOTER,
        film="FILM E1 · THE DUEL",
    )



# E1 opens with a card on how to read it and why it is a fair test, and closes with the final score
E1_CARD_SECONDS = {"intro": 7, "outro": 5}
E1_CLOCK = FILM.get("issue", "DAY−1 18:00").split()[-1]          # e.g. 18:00
E1_CUTOFF = (FILM.get("cutoff") or "").split()[-1:] or [None]  # e.g. 16:00; absent in older inputs
E1_CUTOFF = E1_CUTOFF[0]
try:
    E1_REFIT = f"{pd.Timedelta(FILM['refit_every']).days} days"
except (KeyError, ValueError):
    E1_REFIT = None
E1_FAIR = [
    f"Every forecast was made at {E1_CLOCK} the evening before"
    + (f", from data published up to {E1_CUTOFF}." if E1_CUTOFF else ", from data published by then."),
    f"The models were retrained every {E1_REFIT} on past data only." if E1_REFIT
    else "The models were retrained on past data only, walking forward through time.",
    f"How the models are combined was fixed on the year before ({FILM['validation'][0]:%b %Y} – {FILM['validation'][1]:%b %Y}).",
    "These days were scored once, after everything was fixed: nothing here was tuned on them.",
]
E1_FOOTER = (f"fair test: forecasts made at {E1_CLOCK} the evening before · models retrained on past data only · "
             "combination fixed on the year before · these days scored once")


def swatch_line(fig, x, y, color, style, width, label, size=10.5):
    fig.add_artist(plt.Line2D([x, x + 0.03], [y, y], transform=fig.transFigure, color=color, linestyle=style,
                              linewidth=max(width, 2.0)))
    fig.text(x + 0.04, y, label, fontsize=size, va="center", color=FILM_STYLE["ink"])


def swatch_box(fig, x, y, color, label, size=10.5, w=0.016, h=0.03):
    fig.patches.append(mpatches.Rectangle((x + 0.007, y - h / 2), w, h, transform=fig.transFigure, facecolor=color,
                                          edgecolor=FILM_STYLE["border"], linewidth=0.6))
    fig.text(x + 0.04, y, label, fontsize=size, va="center", color=FILM_STYLE["ink"])


def e1_intro(fig, t):
    fig.clf()
    fig.patch.set_facecolor(FILM_STYLE["paper"])
    fade = lambda a, b: ease(seg(t, a, b))
    fig.text(0.05, 0.92, "GRIDSTRESS  /  FILM E1 · THE DUEL", color=FILM_STYLE["kicker"], fontsize=11, fontweight="bold",
             alpha=fade(0, 0.1))
    fig.text(0.05, 0.83, "Our ensemble against the official forecast", color=FILM_STYLE["ink"], fontsize=27,
             fontweight="bold", alpha=fade(0.02, 0.15))
    fig.text(0.05, 0.775, f"every day from {E1_DAYS[0]:%d %B %Y} to {E1_DAYS[-1]:%d %B %Y} · {len(E1_DAYS)} days · "
             "residual load = consumption − wind − solar, day-ahead", color=FILM_STYLE["muted"], fontsize=12, alpha=fade(0.05, 0.2))

    a = fade(0.15, 0.35)
    fig.text(0.05, 0.68, "HOW TO READ IT", color=FILM_STYLE["kicker2"], fontsize=10.5, fontweight="bold", alpha=a)
    if a > 0:
        y = 0.62
        for _, label, color, style, width in E1_LINES:
            swatch_line(fig, 0.05, y, color, style, width, label)
            y -= 0.058
        swatch_box(fig, 0.05, y, FILM_STYLE["win"], "bar up: the ensemble was closer to the actual that hour")
        swatch_box(fig, 0.05, y - 0.058, FILM_STYLE["loss"], f"bar down: the {DUEL['short']} was closer that hour")
        y -= 0.13
        fig.text(0.05, y, "calendar, one square per day:", fontsize=10.5, color=FILM_STYLE["ink"], va="center")
        for n, (label, color) in enumerate([("ensemble day", FILM_STYLE["win"]), (f"{DUEL['short']} day", FILM_STYLE["loss"]),
                                            (f"tie (±{TIE_MWH:,.0f} MWh)", FILM_STYLE["tie"])]):
            swatch_box(fig, 0.05 + n * 0.135, y - 0.058, color, label, size=10)

    b = fade(0.40, 0.60)
    fig.text(0.56, 0.68, "WHY THIS IS A FAIR TEST", color=FILM_STYLE["win"], fontsize=10.5, fontweight="bold", alpha=b)
    if b > 0:
        fig.patches.append(mpatches.Rectangle((0.545, 0.17), 0.41, 0.48, transform=fig.transFigure,
                                              facecolor=FILM_STYLE["card"], edgecolor=FILM_STYLE["border"], linewidth=1))
        y = 0.60
        for n, line in enumerate(E1_FAIR):
            show = fade(0.42 + n * 0.07, 0.55 + n * 0.07)
            fig.text(0.565, y, f"{n + 1}", fontsize=17, fontweight="bold", color=FILM_STYLE["win"], va="top", alpha=show)
            wrapped = "\n".join(textwrap.wrap(line, 46))
            fig.text(0.595, y, wrapped, fontsize=11, color=FILM_STYLE["ink"], va="top", alpha=show, linespacing=1.35)
            y -= 0.105
    fig.text(0.05, 0.05, "the score builds up day by day: watch the calendar fill and the race settle",
             color=FILM_STYLE["muted"], fontsize=10, alpha=fade(0.8, 0.95))


def e1_outro(fig, t):
    fig.clf()
    fig.patch.set_facecolor(FILM_STYLE["paper"])
    fade = lambda a, b: ease(seg(t, a, b))
    won = int((E1_TABLE["verdict"] == 1).sum())
    lost = int((E1_TABLE["verdict"] == -1).sum())
    tied = int((E1_TABLE["verdict"] == 0).sum())
    hero, opp = float(E1_TABLE["hero_mae"].mean()), float(E1_TABLE["opp_mae"].mean())
    better = 100 * (1 - hero / opp)
    fig.text(0.05, 0.92, "GRIDSTRESS  /  FILM E1 · THE DUEL", color=FILM_STYLE["kicker"], fontsize=11, fontweight="bold")
    fig.text(0.05, 0.83, f"The score, {E1_DAYS[0]:%d %b %Y} – {E1_DAYS[-1]:%d %b %Y}", color=FILM_STYLE["ink"],
             fontsize=27, fontweight="bold", alpha=fade(0, 0.15))
    boxes = [
        ("DAYS WON BY THE ENSEMBLE", f"{won}", f"of {len(E1_DAYS)} days", FILM_STYLE["win"]),
        (f"DAYS WON BY THE {DUEL['short'].upper()}", f"{lost}", f"{tied} ties within ±{TIE_MWH:,.0f} MWh", FILM_STYLE["loss"]),
        ("ENSEMBLE ERROR PER HOUR", f"{hero:,.0f}", f"MWh on average · {DUEL['short']}: {opp:,.0f} MWh", FILM["hero_color"]),
        ("ERROR REDUCED BY", f"{better:.1f} %", f"against the {DUEL['short']}", FILM_STYLE["win"] if better > 0 else FILM_STYLE["loss"]),
    ]
    for n, (title, value, sub, color) in enumerate(boxes):
        show = fade(0.1 + n * 0.12, 0.3 + n * 0.12)
        if show <= 0:
            continue
        x = 0.05 + n * 0.23
        fig.patches.append(mpatches.Rectangle((x, 0.36), 0.21, 0.33, transform=fig.transFigure, facecolor=FILM_STYLE["card"],
                                              edgecolor=FILM_STYLE["border"], linewidth=1, alpha=show))
        fig.text(x + 0.015, 0.645, title, fontsize=fitted(title, 9.5, 0.185 * FILM_SIZE[0] * FILM_DPI, bold=True),
                 fontweight="bold", color=FILM_STYLE["muted"], alpha=show)
        fig.text(x + 0.015, 0.50, value, fontsize=40, fontweight="bold", color=color, alpha=show)
        fig.text(x + 0.015, 0.40, sub, fontsize=10, color=FILM_STYLE["muted"], alpha=show)
    fig.text(0.05, 0.22, "Forecasts made the evening before, models retrained on past data only, the combination fixed on the "
             "year before:\nthese days were never used to build or tune anything shown here.", fontsize=11.5,
             color=FILM_STYLE["ink"], alpha=fade(0.6, 0.8), linespacing=1.4)


E1_FPS = 24
E1_SLOW_DAYS, E1_SLOW_FRAMES, E1_FAST_FRAMES = 14, 12, 2  # frames per day: half a second each for two weeks, then quick
E1_POSITIONS = [d + (s + 1) / n for d in range(len(E1_DAYS))
                for n in [E1_SLOW_FRAMES if d < E1_SLOW_DAYS else E1_FAST_FRAMES] for s in range(n)]
E1_POSITIONS += [float(len(E1_DAYS))] * E1_FPS  # one second on the last day


def film_e1_full(fig, k):
    intro = E1_CARD_SECONDS["intro"] * E1_FPS
    if k < intro:
        return e1_intro(fig, k / max(intro - 1, 1))
    k -= intro
    if k < len(E1_POSITIONS):
        return film_e1(fig, E1_POSITIONS[k])
    k -= len(E1_POSITIONS)
    return e1_outro(fig, k / max(E1_CARD_SECONDS["outro"] * E1_FPS - 1, 1))


E1_FRAMES = (E1_CARD_SECONDS["intro"] + E1_CARD_SECONDS["outro"]) * E1_FPS + len(E1_POSITIONS)



# ---------- E2 · hard-case chase: twelve days, revealed hour by hour ----------
E2_FRAMES_PER_CASE = 18
ranked = FILM["days"]["edge"].sort_values()
E2_CASES = list(ranked.index[::-1][:6]) + list(ranked.index[:6])
E2_CASES = list(dict.fromkeys(E2_CASES))  # a short test window could list a day twice


def film_e2(fig, k):
    fig.clf()
    case, step = divmod(min(k, len(E2_CASES) * E2_FRAMES_PER_CASE - 1), E2_FRAMES_PER_CASE)
    day = E2_CASES[case]
    part = E_HOURS_OF[day]
    shown = part.iloc[: int(np.clip(np.ceil((step + 1) / E2_FRAMES_PER_CASE * len(part)), 2, len(part)))]
    values = part[["actual", "hero", "opp", "other"]]
    pad = (values.max().max() - values.min().min()) * 0.08

    main = fig.add_axes([0.07, 0.33, 0.655, 0.47])
    plain_axes(main)
    forecast_lines(main, shown, shown.index.hour)
    main.set_xlim(-0.5, 23.5)
    main.set_ylim(values.min().min() - pad, values.max().max() + pad)
    main.set_xticks(range(0, 24, 3))
    main.tick_params(labelbottom=False)
    main.set_ylabel("residual load (MWh)", color=FILM_STYLE["muted"], fontsize=8.5)
    mwh_axis(main)

    bars = fig.add_axes([0.07, 0.11, 0.655, 0.19], sharex=main)
    plain_axes(bars)
    limit = max(part["gain"].abs().max(), TIE_MWH) * 1.15
    bars.bar(shown.index.hour, shown["gain"], width=0.8,
             color=np.where(shown["gain"] >= 0, FILM_STYLE["win"], FILM_STYLE["loss"]))
    bars.axhline(0, color=FILM_STYLE["ink"], linewidth=0.8)
    bars.set_ylim(-limit, limit)
    bars.set_xlabel("local hour", color=FILM_STYLE["muted"], fontsize=8.5)
    bars.set_ylabel("gain (MWh)", color=FILM_STYLE["muted"], fontsize=7.5)
    mwh_axis(bars)

    row = FILM["days"].loc[day]
    trophy = row["edge"] > 0
    chrome(
        fig, f"case {case + 1:02d} / {len(E2_CASES)}", f"{day:%a %d %b %Y} · " + ("a trophy" if trophy else "a wreck"),
        f"the six clearest ensemble wins, then the six clearest losses against the {DUEL['short']}",
        [
            ("DAILY EDGE", signed(row["edge"], unit="MWh"), f"{DUEL['short']} MAE − ensemble MAE",
             FILM_STYLE["win"] if trophy else FILM_STYLE["loss"]),
            ("DAY MAE", f"{row['hero_mae']:,.0f} : {row['opp_mae']:,.0f}", f"ensemble : {DUEL['short']} · {DUEL['other_short']} {row['other_mae']:,.0f}", FILM_STYLE["ink"]),
            ("MEMBERS DISAGREE", f"{row['spread']:,.0f} MWh", "mean max − min of the models · known at issue", FILM_STYLE["ink"]),
            ("REGIME", str(row["regime"]), "most hours, from the members' mean forecast", FILM_STYLE["ink"]),
        ],
        line_caption() + " · bars = gain per hour",
        film="FILM E2 · HARD-CASE CHASE",
    )


# ---------- E3 · the mechanism: one point per test day ----------
E3_STEP = 3
E3_X = FILM["days"]["opp_error"].to_numpy()
E3_Y = FILM["days"]["change"].to_numpy()
E3_COLORS = LEVEL_SCALE(FILM["days"]["percentile"].to_numpy())
E3_LIMIT = float(max(np.abs(E3_X).max(), np.abs(E3_Y).max())) * 1.08


def film_e3(fig, k):
    fig.clf()
    n = min(len(E3_X), (k + 1) * E3_STEP)
    ax = fig.add_axes([0.08, 0.11, 0.645, 0.69])
    plain_axes(ax)
    ax.grid(axis="x", color=FILM_STYLE["grid"], linewidth=0.8)
    ax.scatter(E3_X[:n], E3_Y[:n], s=28, c=E3_COLORS[:n], edgecolor="white", linewidth=0.4, zorder=3)
    ax.axhline(0, color=FILM_STYLE["ink"], linewidth=0.8)
    ax.axvline(0, color=FILM_STYLE["ink"], linewidth=0.8)
    ax.set_xlim(-E3_LIMIT, E3_LIMIT)
    ax.set_ylim(-E3_LIMIT, E3_LIMIT)
    ax.set_xlabel(f"{DUEL['short']}'s day error, forecast − actual (MWh)", color=FILM_STYLE["muted"], fontsize=8.5)
    ax.set_ylabel(f"ensemble − {DUEL['short']}, day mean (MWh)", color=FILM_STYLE["muted"], fontsize=8.5)
    mwh_axis(ax)
    mwh_axis(ax, "x")
    ax.plot([-E3_LIMIT, E3_LIMIT], [E3_LIMIT, -E3_LIMIT], color=FILM_STYLE["faint"], linewidth=1, linestyle=(0, (2, 3)))
    ax.text(E3_LIMIT * 0.97, -E3_LIMIT * 0.90, "slope −1: the error fully cancelled", ha="right", fontsize=7.4, color=FILM_STYLE["muted"])
    cards = [("DAYS OBSERVED", f"{n}", f"of {len(E3_X)} full test days", FILM_STYLE["ink"])]
    if n > 10:
        slope, intercept = np.polyfit(E3_X[:n], E3_Y[:n], 1)
        ax.plot([-E3_LIMIT, E3_LIMIT], [slope * -E3_LIMIT + intercept, slope * E3_LIMIT + intercept],
                color=FILM_STYLE["loss"], linewidth=2.2)
        r = float(np.corrcoef(E3_X[:n], E3_Y[:n])[0, 1])
        cards = [
            ("RUNNING CORRELATION", f"{r:+.2f}", f"change against the {DUEL['short']}'s error", FILM_STYLE["loss"]),
            ("FIT SLOPE", f"{slope:+.2f}", f"the mix cancels {max(0.0, -slope):.0%} of that error", FILM_STYLE["ink"]),
        ] + cards + [("MEAN GAIN SO FAR", signed(float(FILM["days"]["edge"].iloc[:n].mean()), unit="MWh"),
                      f"day MAE, {DUEL['short']} − ensemble", FILM_STYLE["win"] if FILM["days"]["edge"].iloc[:n].mean() >= 0 else FILM_STYLE["loss"])]
    chrome(
        fig, "", f"The mix leans against the {DUEL['short']}'s mistake",
        f"one point per full test day · if the ensemble only copied the {DUEL['short']}, every point would sit on 0",
        cards,
        "colour = the day's mean percentile of the actual (green low → red high) · red line = running least-squares fit · "
        f"the y axis contains −x by construction: slope 0 copies the {DUEL['short']}, slope −1 cancels its error",
        film="FILM E3 · THE MECHANISM",
    )


# ---------- E4 · one national day: the day with the largest ensemble edge ----------
E4_DAY = FILM["days"]["edge"].idxmax()
E4_PART = E_HOURS_OF[E4_DAY]


def film_e4(fig, k):
    fig.clf()
    h = min(k, len(E4_PART) - 1)
    shown = E4_PART.iloc[: h + 1]
    values = E4_PART[["actual", "hero", "opp", "other"]]
    pad = (values.max().max() - values.min().min()) * 0.08
    ax = fig.add_axes([0.07, 0.30, 0.655, 0.50])
    plain_axes(ax)
    hours = shown.index.hour
    ax.plot(hours, shown["other"], color=DUEL["other_color"], linewidth=1.0, alpha=0.55, linestyle=(0, (4, 3)))
    ax.plot(hours, shown["opp"], color=DUEL["color"], linewidth=1.8)
    ax.plot(hours, shown["hero"], color=FILM["hero_color"], linewidth=2.0)
    for j in range(len(shown) - 1):
        ax.plot(hours[j: j + 2], shown["actual"].iloc[j: j + 2], color=LEVEL_SCALE(shown["percentile"].iloc[j]),
                linewidth=3.6, solid_capstyle="round")
    ax.set_xlim(-0.5, 23.5)
    ax.set_ylim(values.min().min() - pad, values.max().max() + pad)
    ax.set_xticks(range(0, 24, 3))
    ax.set_xlabel("local hour", color=FILM_STYLE["muted"], fontsize=8.5)
    ax.set_ylabel("residual load (MWh)", color=FILM_STYLE["muted"], fontsize=8.5)
    mwh_axis(ax)

    scale = fig.add_axes([0.07, 0.13, 0.655, 0.045])
    scale.imshow(np.linspace(0, 1, 256)[None, :], aspect="auto", cmap=LEVEL_SCALE, extent=[0, 1, 0, 1])
    scale.set_xlim(0, 1)
    scale.set_ylim(0, 1)
    scale.set_yticks([])
    scale.set_xticks([0, 0.25, 0.5, 0.75, 1], ["P0", "P25", "P50", "P75", "P100"], fontsize=7)
    scale.tick_params(length=0, colors=FILM_STYLE["muted"])
    for side in scale.spines.values():
        side.set_visible(False)
    row = shown.iloc[-1]
    scale.plot([row["percentile"]], [1.45], marker="v", markersize=11, color=FILM_STYLE["ink"], clip_on=False)
    scale.text(0, 1.9, "percentile of this hour's actual among all common test hours", fontsize=7.4,
               color=FILM_STYLE["muted"], transform=scale.transAxes)

    banked = float(shown["gain"].sum())
    chrome(
        fig, "", f"{E4_DAY:%A %d %B %Y} · the year's biggest ensemble day",
        f"{len(E4_PART)} hours · day edge {signed(FILM['days'].at[E4_DAY, 'edge'], unit='MWh')} against the {DUEL['short']}",
        [
            ("ACTUAL RESIDUAL LOAD", f"{row['actual']:,.0f} MWh", f"{shown.index[-1]:%H:%M} local", FILM_STYLE["ink"]),
            (f"{DUEL['short'].upper()} ERROR", signed(row["opp"] - row["actual"], unit="MWh"), "forecast − actual, this hour", DUEL["color"]),
            ("ENSEMBLE ERROR", signed(row["hero"] - row["actual"], unit="MWh"), "forecast − actual, this hour", FILM["hero_color"]),
            ("ERROR AVOIDED", signed(banked, unit="MWh"), f"summed over the hours so far · this hour {signed(row['gain'])}",
             FILM_STYLE["win"] if banked >= 0 else FILM_STYLE["loss"]),
        ],
        "thick trace = actual, coloured by its percentile (green low → red high) · " + line_caption().split(" · ", 1)[1],
        film="FILM E4 · ONE NATIONAL DAY",
    )


# ---------- E5 · the claim: is the edge over the opponent certain? ----------
BOOT_RESAMPLES = 2000  # bootstrap resamples, as in regression-models-hari.ipynb §8
BOOT_BLOCK_DAYS = 7    # days are resampled in blocks of a week: neighbouring days have similar errors


def block_bootstrap(values, rng):
    """Mean and 95 % interval of `values` (one per day) by a circular block bootstrap."""
    values = np.asarray(values, dtype=float)
    n, block = len(values), min(BOOT_BLOCK_DAYS, len(values))
    starts = rng.integers(0, n, size=(BOOT_RESAMPLES, int(np.ceil(n / block))))
    index = (starts[:, :, None] + np.arange(block)[None, None, :]).reshape(BOOT_RESAMPLES, -1)[:, :n] % n
    means = values[index].mean(axis=1)
    return float(values.mean()), float(np.quantile(means, 0.025)), float(np.quantile(means, 0.975))


def claim_of(low, high):
    return "BETTER · certain" if low > 0 else "WORSE · certain" if high < 0 else "UNDECIDED · no claim"


claim_days = FILM["days"].index
claim_hours = FILM["hourly"].index[FILM["hourly"].index.normalize().isin(claim_days)]
day_mae_of = lambda forecast: (forecast.loc[claim_hours] - FILM["hourly"].loc[claim_hours, "actual"]).abs().groupby(claim_hours.normalize()).mean()
opponent_day_mae = day_mae_of(FILM["hourly"]["opp"])
claim_rows = [(meta["label"], meta["color"], FILM["ensembles"][method], method == FILM["hero_method"])
              for method, meta in FILM["ensemble_meta"].items()]
claim_rows += ([(f"{FILM['best_label']} (best member)", FILM["member_color"][FILM["best_member"]], FILM["hourly"]["best"], False)]
               if DUEL_AGAINST == "smard" else [("SMARD day-ahead", smard_color, FILM["hourly"]["smard"], False)])
rng_claim = np.random.default_rng(SEED)
claims = []
for label, color, forecast, is_hero in claim_rows:
    edge = opponent_day_mae - day_mae_of(forecast)  # > 0: the row is better than the opponent that day
    mean, low, high = block_bootstrap(edge.to_numpy(), rng_claim)
    claims.append({"label": label, "color": color, "hero": is_hero, "edge": mean, "low": low, "high": high,
                   "days_better": float((edge > 0).mean()), "claim": claim_of(low, high)})
FILM["claims"] = pd.DataFrame(claims)

# Conditional certainty for the ensemble: quintiles of the members' disagreement, known at issue time
hero_edge = FILM["days"]["edge"]
quintile = pd.qcut(FILM["days"]["spread"], 5, labels=["Q1 agree", "Q2", "Q3", "Q4", "Q5 disagree"])
conditional = []
for name in quintile.cat.categories:
    part = hero_edge[quintile == name]
    mean, low, high = block_bootstrap(part.to_numpy(), rng_claim)
    conditional.append({"quintile": name, "days": len(part), "edge": mean, "low": low, "high": high,
                        "spread": float(FILM["days"]["spread"][quintile == name].mean()), "certain": low > 0})
FILM["conditional"] = pd.DataFrame(conditional).set_index("quintile")


def film_e5(fig, k):
    fig.clf()
    t = k / max(E5_FRAMES - 1, 1)
    claims, conditional = FILM["claims"], FILM["conditional"]
    ax = fig.add_axes([0.25, 0.45, 0.35, 0.35])
    plain_axes(ax)
    ax.grid(axis="x", color=FILM_STYLE["grid"], linewidth=0.8)
    ax.grid(axis="y", visible=False)
    span = max(claims["high"].abs().max(), claims["low"].abs().max(), TIE_MWH) * 1.25
    ax.set_xlim(-span, span)
    ax.set_ylim(len(claims) - 0.5, -0.7)
    ax.set_yticks([])
    ax.axvline(0, color=DUEL["color"], linewidth=1.4, linestyle=(0, (5, 4)))
    ax.text(0, -0.62, f"{DUEL['short']} · 0", ha="center", fontsize=8, color=DUEL["color"], fontweight="bold")
    ax.xaxis.set_major_formatter(lambda value, _: signed(value))
    ax.set_xlabel(f"daily MAE edge over the {DUEL['short']} (MWh), + = better", color=FILM_STYLE["muted"], fontsize=8.5)
    for i, row in claims.iterrows():
        grow = ease(seg(t, 0.04 + 0.09 * i, 0.20 + 0.09 * i))
        color = FILM_STYLE["win"] if row["low"] > 0 else FILM_STYLE["loss"] if row["high"] < 0 else FILM_STYLE["faint"]
        ax.plot([row["edge"] + (row["low"] - row["edge"]) * grow, row["edge"] + (row["high"] - row["edge"]) * grow], [i, i],
                color=color, linewidth=4, alpha=0.55, solid_capstyle="round")
        ax.plot([row["edge"]], [i], "o", markersize=8, color=row["color"], markeredgecolor="white", zorder=3)
        ax.text(-span * 1.03, i, row["label"], ha="right", va="center", fontsize=8.5, clip_on=False,
                color=FILM_STYLE["ink"], fontweight="bold" if row["hero"] else "normal")
        if grow >= 1:
            ax.text(span * 1.03, i, row["claim"], va="center", fontsize=8, clip_on=False, fontweight="bold",
                    color=color if color != FILM_STYLE["faint"] else FILM_STYLE["muted"])
            ax.text(row["edge"], i - 0.32, f"{signed(row['edge'])} [{signed(row['low'])}, {signed(row['high'])}]",
                    ha="center", fontsize=7, color=FILM_STYLE["muted"])

    reveal = ease(seg(t, 0.55, 0.75))
    if reveal > 0:
        strip = stage(fig, [0.045, 0.10, 0.68, 0.22])
        strip.set_xlim(0, 5)
        strip.set_ylim(0, 1)
        strip.text(0.04, 0.97, f"WHEN is the ensemble better for sure? Days split by how much the models disagree, known at issue time",
                   fontsize=8.5, fontweight="bold", color=FILM_STYLE["ink"], alpha=reveal, va="top")
        for i, (name, row) in enumerate(conditional.iterrows()):
            color = FILM_STYLE["win"] if row["certain"] else FILM_STYLE["faint"]
            strip.add_patch(mpatches.Rectangle((i + 0.04, 0.04), 0.92, 0.70, facecolor=color, alpha=0.12 * reveal,
                                               edgecolor=color, linewidth=1))
            strip.text(i + 0.12, 0.60, name, fontsize=8.5, fontweight="bold", color=FILM_STYLE["ink"], alpha=reveal)
            strip.text(i + 0.12, 0.36, signed(row["edge"], unit="MWh"), fontsize=12, fontweight="bold",
                       color=FILM_STYLE["win"] if row["certain"] else FILM_STYLE["muted"], alpha=reveal)
            strip.text(i + 0.12, 0.13, f"[{signed(row['low'])}, {signed(row['high'])}] · {'YES' if row['certain'] else 'not sure'}",
                       fontsize=6.8, color=FILM_STYLE["muted"], alpha=reveal)

    hero = claims[claims["hero"]].iloc[0]
    sure = [name for name, row in conditional.iterrows() if row["certain"]]
    last = ease(seg(t, 0.85, 0.95))
    if last > 0:
        if hero["low"] > 0:
            line = f"{hero['label']} is better than the {DUEL['short']} for sure: the whole 95 % interval lies above 0"
        elif hero["high"] < 0:
            line = f"{hero['label']} is worse than the {DUEL['short']} for sure: the whole 95 % interval lies below 0"
        else:
            line = f"No claim: the 95 % interval of {hero['label']} straddles 0, a mean difference is not a claim"
        fig.text(0.39, 0.355, line, ha="center", fontsize=10.5, fontweight="bold", alpha=last,
                 color=FILM_STYLE["win"] if hero["low"] > 0 else FILM_STYLE["loss"] if hero["high"] < 0 else FILM_STYLE["muted"])
    chrome(
        fig, "", "A mean difference is not a claim",
        f"paired daily MAE edge over the {DUEL['short']} · {len(claim_days)} full test days · 95 % block bootstrap, "
        f"{BOOT_BLOCK_DAYS}-day blocks, {BOOT_RESAMPLES:,} resamples",
        [
            ("ENSEMBLE EDGE", signed(hero["edge"], unit="MWh"), f"per day over the {DUEL['short']}", FILM["hero_color"]),
            ("95 % INTERVAL", f"[{signed(hero['low'])}, {signed(hero['high'])}]", hero["claim"],
             FILM_STYLE["win"] if hero["low"] > 0 else FILM_STYLE["loss"] if hero["high"] < 0 else FILM_STYLE["muted"]),
            ("DAYS BETTER", f"{hero['days_better']:.0%}", f"of {len(claim_days)} full test days", FILM_STYLE["ink"]),
            ("CERTAIN QUINTILES", f"{len(sure)} of 5", ", ".join(sure) if sure else "none on its own", FILM_STYLE["ink"]),
        ],
        "a claim exists only when the whisker clears 0 (teal better, coral worse, grey no claim) · as §8 of "
        "regression-models-hari.ipynb, with week blocks instead of single days · disagreement = members' max − min, day mean",
        film="FILM E5 · THE CLAIM",
    )


E5_FRAMES = 14 * 12  # 14 s at 12 fps


# ---------- E6 · the clock: where in the day the edge comes from ----------
clock = FILM["hourly"].assign(local_hour=FILM["hourly"].index.hour, opp_error=FILM["hourly"]["opp"] - FILM["hourly"]["actual"],
                              change=FILM["hourly"]["hero"] - FILM["hourly"]["opp"])
FILM["clock"] = clock.groupby("local_hour").agg(
    opp_bias=("opp_error", "mean"), change=("change", "mean"), edge=("gain", "mean"), hours=("gain", "size")).reindex(range(24))
E6_FRAMES_PER_HOUR = 8


def film_e6(fig, k):
    fig.clf()
    clock = FILM["clock"]
    h = min(k // E6_FRAMES_PER_HOUR, 23)
    shown = clock.loc[:h]
    top = fig.add_axes([0.07, 0.47, 0.655, 0.31])
    plain_axes(top)
    span = max(clock[["opp_bias", "change"]].abs().max().max(), 1) * 1.25
    top.axhline(0, color=FILM_STYLE["ink"], linewidth=0.8)
    top.plot(clock.index, clock["opp_bias"], color=DUEL["color"], linewidth=1, alpha=0.15)
    top.plot(clock.index, -clock["opp_bias"], color=DUEL["color"], linewidth=1, alpha=0.15, linestyle=(0, (4, 3)))
    top.plot(shown.index, shown["opp_bias"], color=DUEL["color"], linewidth=2.2, marker="o", markersize=3.5)
    top.plot(shown.index, -shown["opp_bias"], color=DUEL["color"], linewidth=1.2, linestyle=(0, (4, 3)))
    top.plot(shown.index, shown["change"], color=FILM["hero_color"], linewidth=2.2, marker="o", markersize=3.5)
    top.axvline(h, color=FILM_STYLE["faint"], linewidth=6, alpha=0.35)
    top.set_xlim(-0.5, 23.5)
    top.set_ylim(-span, span)
    top.set_xticks(range(0, 24, 3))
    top.tick_params(labelbottom=False)
    top.set_ylabel("mean per hour (MWh)", color=FILM_STYLE["muted"], fontsize=8.5)
    mwh_axis(top)
    top.text(0, span * 0.88, f"solid: the {DUEL['short']}'s mean error · dashed: that error mirrored · "
             f"{FILM['hero_label']}: what the ensemble changes", fontsize=7.6, color=FILM_STYLE["muted"])

    bars = fig.add_axes([0.07, 0.12, 0.655, 0.29], sharex=top)
    plain_axes(bars)
    limit = max(clock["edge"].abs().max(), TIE_MWH) * 1.3
    for hour, row in shown.iterrows():
        color = FILM_STYLE["win"] if row["edge"] >= 0 else FILM_STYLE["loss"]
        bars.bar(hour, row["edge"], width=0.72, color=color)
    bars.axhline(0, color=FILM_STYLE["ink"], linewidth=0.8)
    bars.set_ylim(-limit, limit)
    bars.set_xlabel("local hour", color=FILM_STYLE["muted"], fontsize=8.5)
    bars.set_ylabel("MAE edge (MWh)", color=FILM_STYLE["muted"], fontsize=8.5)
    bars.yaxis.set_major_formatter(lambda value, _: signed(value))

    row = clock.loc[h]
    won = int((shown["edge"] > 0).sum())
    mirror = float(np.corrcoef(clock["change"], -clock["opp_bias"])[0, 1]) if clock["opp_bias"].std() > 0 else np.nan
    chrome(
        fig, "", "The clock: where in the day the edge is made",
        f"every common test hour, by local hour · above 0 = ensemble better than the {DUEL['short']}",
        [
            ("HOUR", f"{h:02d}:00", f"{int(row['hours']):,} test hours at this hour", FILM_STYLE["ink"]),
            (f"{DUEL['short'].upper()} BIAS", signed(row["opp_bias"], unit="MWh"), "mean forecast − actual", DUEL["color"]),
            ("ENSEMBLE CHANGE", signed(row["change"], unit="MWh"), f"ensemble − {DUEL['short']}, mean", FILM["hero_color"]),
            ("HOURS AHEAD", f"{won} of {h + 1}", f"change vs mirrored bias: r = {mirror:+.2f}",
             FILM_STYLE["win"] if won * 2 >= h + 1 else FILM_STYLE["loss"]),
        ],
        "as §6 and §9 of regression-models-hari.ipynb · if the ensemble only removed the opponent's bias, its line would "
        "lie on the dashed mirror · bars = MAE edge at that hour",
        film="FILM E6 · THE CLOCK",
    )


E6_FRAMES = 24 * E6_FRAMES_PER_HOUR


# ---------- E7 · the risk days: caught, missed, false alarm (slide 13 of the deck) ----------
# The rules of risk-definition.ipynb, "any" rule: a day is a risk day when one hour crosses that day's line.
# A forecast flags a day when one of its 24 hours crosses the same line, known before the day starts.
RISK_KINDS = {
    "high": {"threshold": "high_threshold_rolling", "actual": "high_risk_rolling_any", "label": "high-risk",
             "line": "high-risk line (top 1 % of the past 365 days)", "color": "#C8553D"},
    "low": {"threshold": "low_threshold_zero", "actual": "low_risk_zero_any", "label": "below-zero",
            "line": "zero line", "color": "#2E64B0"},
}
VERDICT_STYLE = {"caught": FILM_STYLE["win"], "missed": FILM_STYLE["loss"], "false alarm": FILM_STYLE["accent"], "quiet": FILM_STYLE["faint"]}
E7_FRAMES = {"high": 6, "low": 2}  # frames per event day at 6 fps: high-risk days are rare, so they stay longer


def crosses(values, threshold, kind):
    return bool((values >= threshold).any()) if kind == "high" else bool((values <= threshold).any())


def verdict_of(actual, flagged):
    return "caught" if actual and flagged else "missed" if actual else "false alarm" if flagged else "quiet"


E7_ROWS = []
if FILM.get("risk") is not None:
    for day in E_DAYS:
        if day not in FILM["risk"].index:
            continue
        hours = E_HOURS_OF[day]
        row = {"day": day}
        for kind, rule in RISK_KINDS.items():
            threshold, actual = FILM["risk"].at[day, rule["threshold"]], FILM["risk"].at[day, rule["actual"]]
            if pd.isna(threshold) or pd.isna(actual):
                row[kind] = None
                continue
            row[kind] = {
                "threshold": float(threshold), "actual": bool(actual),
                "actual_by_hours": crosses(hours["actual"], threshold, kind),
                "smard": verdict_of(bool(actual), crosses(hours["smard"], threshold, kind)),
                "hero": verdict_of(bool(actual), crosses(hours["hero"], threshold, kind)),
            }
        E7_ROWS.append(row)
E7_ALL = pd.DataFrame(E7_ROWS).set_index("day") if E7_ROWS else pd.DataFrame()
# Only days where something happened: the actual crossed, or a forecast raised an alarm
E7_EVENTS = [
    (day, kind) for day, row in E7_ALL.iterrows() for kind in ("high", "low")
    if row[kind] is not None and (row[kind]["actual"] or "quiet" not in (row[kind]["smard"], row[kind]["hero"]))
]


E7_STARTS = np.cumsum([0] + [E7_FRAMES[kind] for _, kind in E7_EVENTS])  # first frame of each event


def risk_score(kind, until=None):
    """Caught, missed and false alarms per forecast over the test days up to `until`."""
    part = E7_ALL if until is None else E7_ALL.loc[:until]
    found = [entry for entry in part[kind] if entry is not None]
    return {who: {v: sum(entry[who] == v for entry in found) for v in ("caught", "missed", "false alarm")}
            for who in ("smard", "hero")} | {"days": sum(entry["actual"] for entry in found)}


def film_e7(fig, k):
    fig.clf()
    i = min(int(np.searchsorted(E7_STARTS, k, side="right")) - 1, len(E7_EVENTS) - 1)
    day, kind = E7_EVENTS[i]
    rule, entry, hours = RISK_KINDS[kind], E7_ALL.at[day, kind], E_HOURS_OF[day]
    values = hours[["actual", "smard", "hero"]]
    low, high = min(values.min().min(), entry["threshold"]), max(values.max().max(), entry["threshold"])
    pad = (high - low) * 0.10

    ax = fig.add_axes([0.07, 0.42, 0.655, 0.37])
    plain_axes(ax)
    x = hours.index.hour
    ax.axhline(entry["threshold"], color=rule["color"], linewidth=1.4, linestyle=(0, (5, 3)))
    beyond = (low - pad, entry["threshold"]) if kind == "low" else (entry["threshold"], high + pad)
    ax.axhspan(*beyond, color=rule["color"], alpha=0.06)
    ax.text(23.4, entry["threshold"], f"{rule['line']} · {entry['threshold']:,.0f} MWh", ha="right",
            va="bottom" if kind == "high" else "top", fontsize=7.6, color=rule["color"], fontweight="bold")
    ax.plot(x, hours["smard"], color=smard_color, linewidth=1.6, linestyle=SMARD_DASH)
    ax.plot(x, hours["hero"], color=FILM["hero_color"], linewidth=2.0)
    ax.plot(x, hours["actual"], color=FILM["member_color"]["actual"], linewidth=2.4)
    ax.set_xlim(-0.5, 23.5)
    ax.set_ylim(low - pad, high + pad)
    ax.set_xticks(range(0, 24, 3))
    ax.set_xlabel("local hour", color=FILM_STYLE["muted"], fontsize=8.5)
    ax.set_ylabel("residual load (MWh)", color=FILM_STYLE["muted"], fontsize=8.5)
    mwh_axis(ax)

    board = stage(fig, [0.07, 0.04, 0.655, 0.26])
    board.set_xlim(0, 2)
    board.set_ylim(0, 1)
    for col, name in enumerate(("high", "low")):
        score, r = risk_score(name, until=day), RISK_KINDS[name]
        x0 = col + 0.02
        board.text(x0, 0.92, f"{r['label']} days so far: {score['days']}", fontsize=9, fontweight="bold", color=r["color"])
        for j, head in enumerate(("caught", "missed", "false alarm")):
            board.text(x0 + 0.36 + j * 0.2, 0.74, head, fontsize=7.4, color=FILM_STYLE["muted"], ha="center")
        for row_i, (who, label) in enumerate((("smard", "SMARD"), ("hero", "Ensemble"))):
            y = 0.52 - row_i * 0.24
            board.text(x0, y, label, fontsize=9, fontweight="bold",
                       color=FILM_STYLE["ink"] if who == "smard" else FILM["hero_color"], va="center")
            for j, head in enumerate(("caught", "missed", "false alarm")):
                board.text(x0 + 0.36 + j * 0.2, y, f"{score[who][head]}", fontsize=13, fontweight="bold",
                           color=VERDICT_STYLE[head], ha="center", va="center")

    crossed = "crossed the line" if entry["actual"] else "stayed inside"
    chrome(
        fig, f"event {i + 1} / {len(E7_EVENTS)}", f"{day:%a %d %b %Y} · {rule['label']} watch",
        f"did the forecast, known the evening before, see the {rule['label']} hour coming?",
        [
            ("ACTUAL", crossed, f"{rule['label']} rule, any hour", rule["color"] if entry["actual"] else FILM_STYLE["muted"]),
            ("SMARD", entry["smard"], "official day-ahead", VERDICT_STYLE[entry["smard"]] if entry["smard"] != "quiet" else FILM_STYLE["muted"]),
            ("ENSEMBLE", entry["hero"], FILM["hero_label"], VERDICT_STYLE[entry["hero"]] if entry["hero"] != "quiet" else FILM_STYLE["muted"]),
            ("THIS DAY'S LINE", f"{entry['threshold']:,.0f} MWh", rule["line"], rule["color"]),
        ],
        "ink = actual · dashed grey = SMARD · ensemble in its colour · a forecast flags the day when one of its hours "
        "crosses the line · labels from risk-definition.ipynb · only days with a risk hour or an alarm are shown",
        film="FILM E7 · THE RISK DAYS",
    )


def hold(frames, fps):
    """Frame count with the last frame held for FILM_HOLD seconds."""
    return frames + FILM_HOLD * fps


FILMS = {
    "film_VI_the_second_glass": {"draw": lambda fig, k: draw_at(fig, k / FILM_FPS), "frames": int(FILM_SECONDS * FILM_FPS), "fps": FILM_FPS},
    "film_E1_the_duel": {"draw": film_e1_full, "frames": E1_FRAMES, "fps": E1_FPS, "dpi": 150},  # 1920 × 1080
    "film_E2_hard_case_chase": {"draw": film_e2, "frames": hold(len(E2_CASES) * E2_FRAMES_PER_CASE, 10), "fps": 10},
    "film_E3_the_mechanism": {"draw": film_e3, "frames": hold(int(np.ceil(len(E3_X) / E3_STEP)), 14), "fps": 14},
    "film_E4_one_national_day": {"draw": film_e4, "frames": hold(len(E4_PART), 4), "fps": 4},
    "film_E5_the_claim": {"draw": film_e5, "frames": hold(E5_FRAMES, 12), "fps": 12},
    "film_E6_the_clock": {"draw": film_e6, "frames": hold(E6_FRAMES, 12), "fps": 12},
}
if E7_EVENTS:
    FILMS["film_E7_the_risk_days"] = {"draw": film_e7, "frames": hold(int(E7_STARTS[-1]), 6), "fps": 6}
else:
    print("E7 skipped: the film inputs hold no risk labels; re-run §8 of ensemble-claude.ipynb")


def frame_image(name, k, dpi=60):
    """Frame `k` of a film as an image array."""
    fig = plt.figure(figsize=FILM_SIZE, dpi=dpi)
    FILMS[name]["draw"](fig, k)
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=dpi)
    plt.close(fig)
    buffer.seek(0)
    return plt.imread(buffer)


# Contact sheet: two frames per companion film, one mid-way and the last
sheet = [(name, k) for name in list(FILMS)[1:] for k in (FILMS[name]["frames"] // 2, FILMS[name]["frames"] - 1)]
fig, axes = plt.subplots(len(sheet) // 2, 2, figsize=(16, 4.6 * len(sheet) // 2))
for ax, (name, k) in zip(axes.ravel(), sheet):
    ax.imshow(frame_image(name, k))
    ax.set_axis_off()
fig.tight_layout(pad=0.4)
plt.show()
for name, film in FILMS.items():
    print(f"{name:<28}: {film['frames']:>4} frames at {film['fps']:>2} fps = {film['frames'] / film['fps']:5.1f} s")

## <a id="sec-4"></a>4 Render

Every film in `FILMS_TO_RENDER` is written to `data/films/` as an mp4, then §5 plays it here. A film is
rendered again only when it is out of date: when the film inputs or a setting it depends on
(`DUEL_AGAINST`, `TIE_MWH`, `FILM_FPS`, the bootstrap, the frame size) changed since its last render.
Each mp4 keeps a small `.json` beside it with those settings. After a change in the drawing code, set
`FORCE_RENDER = True` once. The first full render takes several minutes and needs ffmpeg
(`imageio-ffmpeg` or a system ffmpeg).

In [ ]:
if RENDER_FILMS:
    # ffmpeg writes the mp4s: a system ffmpeg on the PATH, or the one shipped with imageio-ffmpeg
    if not animation.FFMpegWriter.isAvailable():
        try:
            import imageio_ffmpeg
        except ImportError:
            print("no ffmpeg found: installing imageio-ffmpeg into this kernel ...", flush=True)
            subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "imageio-ffmpeg"], check=True)
            import imageio_ffmpeg
        plt.rcParams["animation.ffmpeg_path"] = imageio_ffmpeg.get_ffmpeg_exe()
    if not animation.FFMpegWriter.isAvailable():
        raise RuntimeError("RENDER_FILMS needs ffmpeg: pip install imageio-ffmpeg, or install ffmpeg on the PATH")
    FILM_DIR.mkdir(exist_ok=True)
    # What a film depends on besides the drawing code: the inputs and the settings that change a frame
    RENDER_STAMP = {
        "inputs_saved_at": INPUTS["saved_at"], "duel_against": DUEL_AGAINST, "tie_mwh": TIE_MWH, "fps": FILM_FPS,
        "boot": [BOOT_RESAMPLES, BOOT_BLOCK_DAYS], "e1_window": str(E1_WINDOW), "look": 4, "colours": "v2", "size": [*FILM_SIZE, FILM_DPI],
    }
    for name in [name for name in FILMS_TO_RENDER if name in FILMS]:
        film, path = FILMS[name], FILM_DIR / f"{name}.mp4"
        stamp_path = path.with_suffix(".json")
        if (not FORCE_RENDER and path.exists() and stamp_path.exists()
                and json.loads(stamp_path.read_text(encoding="utf-8")) == RENDER_STAMP):
            print(f"up to date: {path.relative_to(DATA_DIR.parent).as_posix()}")
            continue
        started = time.perf_counter()
        fig = plt.figure(figsize=FILM_SIZE, dpi=film.get("dpi", FILM_DPI))
        movie = animation.FuncAnimation(fig, lambda k, draw=film["draw"]: draw(fig, k), frames=film["frames"],
                                        interval=1000 / film["fps"])
        movie.save(path, writer=animation.FFMpegWriter(fps=film["fps"], bitrate=6000 if film.get("dpi", FILM_DPI) > FILM_DPI else 2400,
                                                       extra_args=["-pix_fmt", "yuv420p"]), dpi=film.get("dpi", FILM_DPI))
        plt.close(fig)
        stamp_path.write_text(json.dumps(RENDER_STAMP, indent=1), encoding="utf-8")
        print(f"wrote {path.relative_to(DATA_DIR.parent).as_posix()}: {film['frames']} frames, "
              f"{film['frames'] / film['fps']:.0f} s, {path.stat().st_size / 1e6:.1f} MB in {time.perf_counter() - started:.0f} s")
else:
    print("films not rendered (RENDER_FILMS = False); §5 plays any mp4s already in "
          f"{FILM_DIR.relative_to(DATA_DIR.parent).as_posix()}/")

## <a id="sec-5"></a>5 Watch

The rendered films, played here in the notebook, in the order of §3. The mp4s live in `data/films/`
(gitignored); this cell plays whatever is there, so a film rendered in an earlier session plays without
rendering again. Each film shows when its file was written: after a change above, render it again
(`RENDER_FILMS = True`) before trusting what it shows.

`SHOW_FILMS = "auto"` (the default) embeds the films in the notebook when it runs in VS Code, which
cannot play files from disk, and links them otherwise (Jupyter, JupyterLab), which keeps the notebook
small. `"embed"` forces embedding everywhere (about 1–8 MB per film), `"link"` forces links.

In [ ]:
import os
from datetime import datetime
from IPython.display import HTML, Markdown

if SHOW_FILMS not in ("auto", "link", "embed", None):
    raise ValueError(f'SHOW_FILMS must be "auto", "link", "embed" or None, not {SHOW_FILMS!r}')
in_vscode = any(key.startswith("VSCODE_") for key in os.environ)
show_mode = ("embed" if in_vscode else "link") if SHOW_FILMS == "auto" else SHOW_FILMS
FILM_TITLES = {
    "film_VI_the_second_glass": "Film VI · The second glass",
    "film_E1_the_duel": "E1 · The duel",
    "film_E2_hard_case_chase": "E2 · Hard-case chase",
    "film_E3_the_mechanism": "E3 · The mechanism",
    "film_E4_one_national_day": "E4 · One national day",
    "film_E5_the_claim": "E5 · The claim",
    "film_E6_the_clock": "E6 · The clock",
    "film_E7_the_risk_days": "E7 · The risk days",
}
shown, missing = [], []
if show_mode:
    for name in FILMS:
        path = FILM_DIR / f"{name}.mp4"
        if not path.exists() or path.stat().st_size == 0:
            missing.append(name)
            continue
        written = datetime.fromtimestamp(path.stat().st_mtime)
        display(Markdown(f"**{FILM_TITLES.get(name, name)}** · `{path.relative_to(DATA_DIR.parent).as_posix()}` · "
                         f"{path.stat().st_size / 1e6:.1f} MB · rendered {written:%d %b %Y %H:%M}"))
        if show_mode == "embed":
            display(Video(str(path), embed=True, width=880, html_attributes="controls loop"))
        else:
            # Jupyter serves files relative to the notebook's folder, so the link must be relative too
            display(Video(Path(os.path.relpath(path, Path.cwd())).as_posix(), width=880, html_attributes="controls loop"))
        shown.append(name)
    if missing:
        print(f"not rendered yet: {', '.join(missing)} · set RENDER_FILMS = True in §1.1 (FILMS_TO_RENDER picks which)")
    print(f"{len(shown)} of {len(FILMS)} films shown ({show_mode}{" · VS Code" if in_vscode else ""})")
else:
    print("SHOW_FILMS = None: films are not played here")

## <a id="sec-6"></a>6 Self-check

In [ ]:
# Every glass and every adaptive day holds weights >= 0 summing to 1
for regime in FILM["regimes"]:
    assert np.isclose(regime["weights"].sum(), 1) and (regime["weights"] >= -1e-12).all(), regime["name"]
assert np.isclose(FILM["global_weights"].sum(), 1)
if FILM["adaptive"] is not None:
    assert np.allclose(FILM["adaptive"]["weights"].sum(axis=1), 1)

# The duel: the hourly gain averages to the MAE difference; the ensemble shown is among the ensembles
hourly = FILM["hourly"]
assert np.isclose(hourly["gain"].mean(), (hourly["opp"] - hourly["actual"]).abs().mean() - (hourly["hero"] - hourly["actual"]).abs().mean())
assert np.isclose((hourly["hero"] - hourly["actual"]).abs().mean(), FILM["hero"]["MAE"]), "inputs disagree with their own scoreboard"
assert np.allclose(FILM["ensembles"][FILM["hero_method"]], hourly["hero"])
assert FILM["days"].index.isin(FILM["test_days"]).all()
if DUEL_AGAINST == "best":
    assert np.allclose(FILM["days"]["edge"], FILM["daily"]["edge"])
assert E4_DAY == FILM["days"]["edge"].idxmax() and len(E2_CASES) <= 12

# E5: intervals contain their mean; E6: the hourly edges add up to the overall gain
assert ((FILM["claims"]["low"] <= FILM["claims"]["edge"]) & (FILM["claims"]["edge"] <= FILM["claims"]["high"])).all()
assert FILM["conditional"]["days"].sum() == len(FILM["days"])
assert np.isclose((FILM["clock"]["edge"] * FILM["clock"]["hours"]).sum() / FILM["clock"]["hours"].sum(), hourly["gain"].mean())

# E7: the risk labels agree with the hours they were made from
if E7_EVENTS:
    for kind in ("high", "low"):
        found = [e for e in E7_ALL[kind] if e is not None]
        assert all(e["actual"] == e["actual_by_hours"] for e in found), f"{kind} risk labels disagree with the actual hours"

# The timeline covers the film; rendered films exist
assert FILM_SECONDS == sum(seconds for _, _, seconds in FILM_ACTS) and FILM_SECONDS > 0
if RENDER_FILMS:
    for name in [name for name in FILMS_TO_RENDER if name in FILMS]:
        assert (FILM_DIR / f"{name}.mp4").stat().st_size > 0, name

print("film self-check passed")
print(f"  {len(FILM['regimes'])} regime glasses + global sum to 1; Film VI {len(FILM_ACTS)} acts, {FILM_SECONDS} s")
print(f"  E1–E7, duel against {FILM['duel']['label']}: hourly gain averages to the MAE difference; "
      f"{len(E_DAYS)} full test days, {len(E2_CASES)} hard cases")